# Thesis Results Plot Generation

Self-contained Colab notebook for regenerating the thesis result plots and summary table. The figures are displayed inline and saved as vector PDFs in `thesis_results_outputs/figures`.

In [ ]:

from __future__ import annotations

import copy
import os
import pickle
import shutil
import sys
import tempfile
from pathlib import Path

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
MPLCONFIGDIR = Path(tempfile.gettempdir()) / "thesis_results_matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from IPython.display import display

OUTPUT_DIR = BASE_DIR / "thesis_results_outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
DATA_DIR = BASE_DIR / "data"

for directory in [FIGURE_DIR, TABLE_DIR, CACHE_DIR, DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.size": 10,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
    }
)

METHODS = ["bp", "np", "np_fan_in", "np_fixed", "wp"]
PERTURBATION_METHODS = ["np", "np_fan_in", "np_fixed", "wp"]
METHOD_LABELS = {
    "bp": "BP",
    "np": "IS-NP",
    "np_fan_in": "Fan-in NP",
    "np_fixed": "Vanilla NP",
    "wp": "WP",
}
METHOD_COLORS = {
    "bp": "#1f77b4",
    "np": "#ff7f0e",
    "np_fan_in": "#9467bd",
    "np_fixed": "#d62728",
    "wp": "#2ca02c",
}

CURVE_LINE_WIDTH = 1.15
LEGEND_LINE_WIDTH = 1.5
CONVERGENCE_FRACTION = 0.90
SUMMARY_SIGNIFICANT_FIGURES = 3

USE_CACHE = True
FORCE_RECOMPUTE = False
VARIANCE_COLUMN = "sample_variance"  # Alternatives: "sample_variance" or "batch_variance".
COSINE_COLUMN = "mean_estimate_cosine"  # Alternative: "avg_sample_cosine".

print(f"Base directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Figures: {FIGURE_DIR}")
print(f"Tables: {TABLE_DIR}")


## Embedded Task Implementations

This notebook is meant to run by itself in Colab. The next cell contains the setup and function definitions copied from the four task notebooks as embedded Python source strings. `load_embedded_task_api(task_key)` executes the relevant source in an isolated namespace so each task keeps its original model, data loader, training loop, and diagnostic functions without needing extra files in Colab.

In [ ]:
EMBEDDED_TASK_SOURCE_CODE = {
    'sinus': 'import itertools\nimport math\nimport random\nimport time\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom IPython.display import display\nfrom torch.utils.data import DataLoader, TensorDataset, Subset\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nif hasattr(torch, "set_float32_matmul_precision"):\n    torch.set_float32_matmul_precision("high")\n\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\nset_seed(0)\nprint(f"device={device}")\nif device.type == "cuda":\n    print(torch.cuda.get_device_name(0))\n\n\nMETHODS = ["bp", "np", "np_fan_in", "np_fixed", "wp"]\nMETHOD_LABELS = {\n    "bp": "BP",\n    "np": "IS-NP",\n    "np_fan_in": "Fan-in NP",\n    "np_fixed": "Vanilla NP",\n    "wp": "WP",\n}\nMETHOD_COLORS = {\n    "bp": "C0",\n    "np": "C1",\n    "np_fan_in": "C4",\n    "np_fixed": "C3",\n    "wp": "C2",\n}\n\n\nclass MLP(nn.Module):\n    def __init__(self, dimensions, activation=torch.sigmoid, output_activation=None, require_grad=False):\n        super().__init__()\n        self.layers = nn.ModuleList(\n            [nn.Linear(dimensions[i], dimensions[i + 1], bias=True) for i in range(len(dimensions) - 1)]\n        )\n        self.activation = activation\n        self.output_activation = output_activation\n        if not require_grad:\n            for p in self.parameters():\n                p.requires_grad_(False)\n\n    def forward(self, x):\n        final_layer = len(self.layers) - 1\n        h = x\n        for i, layer in enumerate(self.layers):\n            u = layer(h)\n            if i == final_layer:\n                h = self.output_activation(u) if self.output_activation else u\n            else:\n                h = self.activation(u)\n        return h\n\n    def forward_weight_perturb(self, x, sigma):\n        batch_size = x.shape[0]\n        xs = [x]\n        ys = []\n        noises = []\n        h = x\n        for i, layer in enumerate(self.layers):\n            w = layer.weight\n            b = layer.bias\n            eps_w = torch.randn(batch_size, *w.shape, device=w.device, dtype=w.dtype) * sigma\n            eps_b = torch.randn(batch_size, *b.shape, device=b.device, dtype=b.dtype) * sigma\n            w_used = w.unsqueeze(0) + eps_w\n            b_used = b.unsqueeze(0) + eps_b\n            u = torch.bmm(h.unsqueeze(1), w_used.transpose(1, 2)).squeeze(1) + b_used\n            final_layer = len(self.layers) - 1\n            if i == final_layer:\n                h = self.output_activation(u) if self.output_activation else u\n                ys.append(h)\n            else:\n                h = self.activation(u)\n                ys.append(h)\n                xs.append(h)\n            noises.append((eps_w, eps_b))\n        return ys, xs, noises\n\n    def forward_node_perturb(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            x_in = a_noisy\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            noise_scale = sigma * torch.sqrt(1.0 + x_in.pow(2).sum(dim=1, keepdim=True))\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n    def forward_node_perturb_fixed_sigma(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            noise_scale = torch.full_like(z_noisy, sigma)\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n\n    def forward_node_perturb_fan_in_scaled(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            fan_in_with_bias = layer.in_features + 1\n            noise_scale_value = sigma * math.sqrt(fan_in_with_bias)\n            noise_scale = torch.full_like(z_noisy, noise_scale_value)\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n\ndef _mean_loss_per_sample(prediction, target):\n    loss = F.mse_loss(prediction, target, reduction="none")\n    if loss.dim() > 1:\n        loss = loss.mean(dim=1)\n    return loss.view(-1)\n\n\ndef _centered_reward_signal(loss_per_sample):\n    reward = -loss_per_sample\n    return reward - reward.mean()\n\n\ndef _flatten_parameter_tensors(weight_tensors, bias_tensors):\n    pieces = []\n    for weight_tensor, bias_tensor in zip(weight_tensors, bias_tensors):\n        pieces.append(weight_tensor.reshape(-1))\n        pieces.append(bias_tensor.reshape(-1))\n    return torch.cat(pieces)\n\n\ndef node_perturbation_step(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef node_perturbation_step_fixed_sigma(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fixed_sigma(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef node_perturbation_step_fan_in_scaled(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fan_in_scaled(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef weight_perturb_step(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    layer_outputs, _, noises = model.forward_weight_perturb(X, sigma)\n    prediction_noisy = layer_outputs[-1]\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    noise_scale = sigma ** 2 + 1e-12\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, (weight_noise, bias_noise) in zip(model.layers, noises):\n            scaled_weight_noise = scalar_signal.view(-1, 1, 1) * weight_noise / noise_scale\n            scaled_bias_noise = scalar_signal.view(-1, 1) * bias_noise / noise_scale\n            raw_weight_update = scaled_weight_noise.mean(dim=0)\n            raw_bias_update = scaled_bias_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef backprop_step(model, X, target, optimizer, loss_fn=F.mse_loss, return_unscaled_parameter_update_vector=False):\n    model.train()\n    for p in model.parameters():\n        p.requires_grad_(True)\n    optimizer.zero_grad()\n    y_pred = model(X)\n    loss = loss_fn(y_pred, target, reduction="mean")\n    loss.backward()\n    weight_grads = [layer.weight.grad.detach().clone() for layer in model.layers]\n    bias_grads = [layer.bias.grad.detach().clone() for layer in model.layers]\n    optimizer.step()\n    if return_unscaled_parameter_update_vector:\n        return loss.item(), -_flatten_parameter_tensors(weight_grads, bias_grads)\n    return loss.item()\n\n\ndef model_weight_norm(model):\n    total = torch.tensor(0.0, device=next(model.parameters()).device)\n    for layer in model.layers:\n        total = total + layer.weight.detach().pow(2).sum()\n    return float(torch.sqrt(total))\n\n\ndef mean_layer_output_norms(model, loader, device, normalize_by_sqrt_fan_in=False):\n    model.eval()\n    summed_norms = [0.0 for _ in model.layers]\n    total_examples = 0\n\n    with torch.no_grad():\n        for batch in loader:\n            if len(batch) == 3:\n                xb, _, _ = batch\n            else:\n                xb, _ = batch\n            xb = xb.to(device, non_blocking=True)\n            h = xb\n            batch_size = xb.size(0)\n\n            for layer_idx, layer in enumerate(model.layers):\n                h_for_norm = h if h.dim() > 1 else h.unsqueeze(1)\n                current_norm = torch.norm(h_for_norm, dim=1)\n                if normalize_by_sqrt_fan_in:\n                    current_norm = current_norm / math.sqrt(h_for_norm.shape[1] + 1)\n                summed_norms[layer_idx] += current_norm.sum().item()\n                u = layer(h)\n                if layer_idx == len(model.layers) - 1:\n                    h = model.output_activation(u) if model.output_activation else u\n                else:\n                    h = model.activation(u)\n\n            total_examples += batch_size\n\n    return [value / max(total_examples, 1) for value in summed_norms]\n\n\ndef mean_normalized_layer_output_norms(model, loader, device):\n    return mean_layer_output_norms(model, loader, device, normalize_by_sqrt_fan_in=True)\n\n\ndef mean_raw_layer_output_norms(model, loader, device):\n    return mean_layer_output_norms(model, loader, device, normalize_by_sqrt_fan_in=False)\n\n\ndef _random_subset_indices(n, limit, seed):\n    if limit is None or limit >= n:\n        return torch.arange(n)\n    g = torch.Generator().manual_seed(seed)\n    return torch.randperm(n, generator=g)[:limit]\n\n\ndef _maybe_subset_dataset(dataset, limit, seed):\n    if limit is None or limit >= len(dataset):\n        return dataset\n    indices = _random_subset_indices(len(dataset), limit, seed)\n    return Subset(dataset, indices.tolist())\n\n\ndef regression_r2_score(y_true, y_pred):\n    ss_res = torch.sum((y_true - y_pred) ** 2)\n    target_mean = torch.mean(y_true, dim=0, keepdim=True)\n    ss_tot = torch.sum((y_true - target_mean) ** 2)\n    if float(ss_tot) < 1e-12:\n        return 0.0\n    return float(1.0 - ss_res / (ss_tot + 1e-12))\n\n\ndef evaluate_model(model, loader, device):\n    model.eval()\n    total_loss = 0.0\n    total_examples = 0\n    predictions = []\n    targets = []\n    with torch.no_grad():\n        for xb, yb in loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            pred = model(xb)\n            loss = F.mse_loss(pred, yb, reduction="mean")\n            total_loss += loss.item() * xb.size(0)\n            total_examples += xb.size(0)\n            predictions.append(pred.detach().cpu())\n            targets.append(yb.detach().cpu())\n    mean_loss = total_loss / max(total_examples, 1)\n    y_pred = torch.cat(predictions, dim=0)\n    y_true = torch.cat(targets, dim=0)\n    r2 = regression_r2_score(y_true, y_pred)\n    return mean_loss, r2\n\n\ndef predict_tensor(model, x, device):\n    model.eval()\n    with torch.no_grad():\n        return model(x.to(device)).detach().cpu()\n\n\ndef make_model(dimensions, require_grad, device):\n    return MLP(dimensions, activation=torch.sigmoid, require_grad=require_grad).to(device)\n\n\ndef build_initial_state_dict(dimensions, seed, device, first_layer_weight_scale=1.0):\n    set_seed(seed)\n    base_model = make_model(dimensions, require_grad=True, device=device)\n    state_dict = {name: tensor.detach().clone() for name, tensor in base_model.state_dict().items()}\n    if first_layer_weight_scale != 1.0:\n        state_dict["layers.0.weight"] = state_dict["layers.0.weight"] * first_layer_weight_scale\n    del base_model\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return state_dict\n\n\ndef train_one_run(\n    method,\n    run_config,\n    data,\n    dimensions,\n    epochs,\n    seed,\n    device,\n    print_every_epoch=1,\n    divergence_loss_threshold=5.0,\n    first_layer_weight_scale=1.0,\n):\n    base_state = build_initial_state_dict(\n        dimensions=dimensions,\n        seed=seed,\n        device=device,\n        first_layer_weight_scale=first_layer_weight_scale,\n    )\n\n    require_grad = method == "bp"\n    model = make_model(dimensions, require_grad=require_grad, device=device)\n    model.load_state_dict(base_state)\n\n    optimizer = None\n    if method == "bp":\n        optimizer = torch.optim.SGD(model.parameters(), lr=run_config["lr"])\n\n    history = {\n        "epoch": [],\n        "train_loss": [],\n        "train_r2": [],\n        "test_loss": [],\n        "test_r2": [],\n        "weight_norm": [],\n        "layer_output_norms": [],\n        "layer_output_norms_raw": [],\n    }\n\n    diverged = False\n    best_test_loss = float("inf")\n    best_test_r2 = -float("inf")\n    start_time = time.time()\n\n    train_loader = data["train_loader"]\n    train_eval_loader = data["train_eval_loader"]\n    test_loader = data["test_loader"]\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        for xb, yb in train_loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n\n            if method == "bp":\n                backprop_step(model, xb, yb, optimizer=optimizer)\n            elif method == "np":\n                node_perturbation_step(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "np_fan_in":\n                node_perturbation_step_fan_in_scaled(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "np_fixed":\n                node_perturbation_step_fixed_sigma(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "wp":\n                weight_perturb_step(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            else:\n                raise ValueError(f"Unknown method: {method}")\n\n        train_loss, train_r2 = evaluate_model(model, train_eval_loader, device)\n        test_loss, test_r2 = evaluate_model(model, test_loader, device)\n\n        history["epoch"].append(epoch)\n        history["train_loss"].append(train_loss)\n        history["train_r2"].append(train_r2)\n        history["test_loss"].append(test_loss)\n        history["test_r2"].append(test_r2)\n        history["weight_norm"].append(model_weight_norm(model))\n        history["layer_output_norms"].append(mean_normalized_layer_output_norms(model, train_eval_loader, device))\n        history["layer_output_norms_raw"].append(mean_raw_layer_output_norms(model, train_eval_loader, device))\n\n        best_test_loss = min(best_test_loss, test_loss)\n        best_test_r2 = max(best_test_r2, test_r2)\n\n        if (not math.isfinite(train_loss)) or (not math.isfinite(test_loss)) or test_loss > divergence_loss_threshold:\n            diverged = True\n            print(\n                f"    diverged at epoch {epoch}/{epochs} | "\n                f"train_loss={train_loss:.4f}, test_loss={test_loss:.4f}, test_r2={test_r2:.4f}"\n            )\n            break\n\n        if epoch % print_every_epoch == 0 or epoch == 1 or epoch == epochs:\n            sigma_str = f", sigma={run_config[\'sigma\']:.4g}" if "sigma" in run_config else ""\n            print(\n                f"    epoch {epoch:2d}/{epochs} | {method} | lr={run_config[\'lr\']:.4g}{sigma_str} | "\n                f"train_loss={train_loss:.4f}, train_r2={train_r2:.4f}, "\n                f"test_loss={test_loss:.4f}, test_r2={test_r2:.4f}"\n            )\n\n    duration_sec = time.time() - start_time\n    state_dict_cpu = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n    result = {\n        "method": method,\n        "lr": float(run_config["lr"]),\n        "sigma": float(run_config["sigma"]) if "sigma" in run_config else np.nan,\n        "epochs_completed": len(history["epoch"]),\n        "final_train_loss": history["train_loss"][-1] if history["train_loss"] else np.nan,\n        "final_train_r2": history["train_r2"][-1] if history["train_r2"] else np.nan,\n        "final_test_loss": history["test_loss"][-1] if history["test_loss"] else np.nan,\n        "final_test_r2": history["test_r2"][-1] if history["test_r2"] else np.nan,\n        "best_test_loss": best_test_loss,\n        "best_test_r2": best_test_r2,\n        "diverged": diverged,\n        "duration_sec": duration_sec,\n        "history": history,\n        "state_dict": state_dict_cpu,\n    }\n    del model\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return result\n\n\ndef true_gradient_vector(model, xb, yb):\n    requires_grad_state = [parameter.requires_grad for parameter in model.parameters()]\n    for parameter in model.parameters():\n        parameter.requires_grad_(True)\n    model.zero_grad(set_to_none=True)\n    prediction = model(xb)\n    loss = F.mse_loss(prediction, yb, reduction="mean")\n    loss.backward()\n    weight_grads = [layer.weight.grad.detach().clone() for layer in model.layers]\n    bias_grads = [layer.bias.grad.detach().clone() for layer in model.layers]\n    flat_grad = _flatten_parameter_tensors(weight_grads, bias_grads)\n    model.zero_grad(set_to_none=True)\n    for parameter, old_value in zip(model.parameters(), requires_grad_state):\n        parameter.requires_grad_(old_value)\n    return -flat_grad\n\n\ndef cosine_similarity_safe(a, b, eps=1e-12):\n    a_norm = torch.norm(a)\n    b_norm = torch.norm(b)\n    if a_norm.item() < eps or b_norm.item() < eps:\n        return 0.0\n    return float(torch.dot(a, b) / (a_norm * b_norm + eps))\n\n\ndef node_perturbation_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef node_perturbation_fan_in_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fan_in_scaled(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef node_perturbation_fixed_sigma_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fixed_sigma(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef weight_perturbation_gradient_estimate_vector(model, xb, yb, sigma):\n    layer_outputs, _, noises = model.forward_weight_perturb(xb, sigma)\n    prediction_noisy = layer_outputs[-1]\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    noise_scale = sigma ** 2 + 1e-12\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for weight_noise, bias_noise in noises:\n        scaled_weight_noise = scalar_signal.view(-1, 1, 1) * weight_noise / noise_scale\n        scaled_bias_noise = scalar_signal.view(-1, 1) * bias_noise / noise_scale\n        raw_weight_updates.append(scaled_weight_noise.mean(dim=0))\n        raw_bias_updates.append(scaled_bias_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef perturbation_gradient_estimate_vector(method, model, xb, yb, sigma):\n    if method == "np":\n        return node_perturbation_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "np_fan_in":\n        return node_perturbation_fan_in_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "np_fixed":\n        return node_perturbation_fixed_sigma_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "wp":\n        return weight_perturbation_gradient_estimate_vector(model, xb, yb, sigma)\n    raise ValueError(f"Unknown perturbation method: {method}")\n\n\ndef variance_against_true_gradient(estimate_matrix, true_update):\n    num_samples = estimate_matrix.shape[0]\n    if num_samples == 0:\n        return 0.0, 0.0\n\n    squared_errors = (estimate_matrix - true_update.unsqueeze(0)).pow(2)\n    per_sample_variance = float(squared_errors.mean(dim=1).mean())\n    mean_estimate = estimate_matrix.mean(dim=0)\n    batch_variance = float((mean_estimate - true_update).pow(2).mean())\n    return per_sample_variance, batch_variance\n\n\ndef train_backprop_checkpoint_states(\n    data,\n    dimensions,\n    epochs,\n    seed,\n    device,\n    lr,\n    checkpoint_epochs,\n    print_every_epoch=100,\n    first_layer_weight_scale=1.0,\n):\n    checkpoint_epochs = sorted(set(int(epoch) for epoch in checkpoint_epochs))\n    initial_state = build_initial_state_dict(\n        dimensions=dimensions,\n        seed=seed,\n        device=device,\n        first_layer_weight_scale=first_layer_weight_scale,\n    )\n    model = make_model(dimensions, require_grad=True, device=device)\n    model.load_state_dict(initial_state)\n    optimizer = torch.optim.SGD(model.parameters(), lr=lr)\n    checkpoint_states = {}\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        for xb, yb in data["train_loader"]:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            backprop_step(model, xb, yb, optimizer=optimizer)\n\n        if epoch in checkpoint_epochs:\n            checkpoint_states[epoch] = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n\n        if epoch % print_every_epoch == 0 or epoch == 1 or epoch == epochs or epoch in checkpoint_epochs:\n            train_loss, train_r2 = evaluate_model(model, data["train_eval_loader"], device)\n            test_loss, test_r2 = evaluate_model(model, data["test_loader"], device)\n            print(\n                f"    bp checkpoint training epoch {epoch:4d}/{epochs} | "\n                f"train_loss={train_loss:.4f}, train_r2={train_r2:.4f}, "\n                f"test_loss={test_loss:.4f}, test_r2={test_r2:.4f}"\n            )\n\n    return checkpoint_states\n\n\ndef plot_raw_layer_output_norm_histories(results_by_method):\n    first_result = next((results_by_method[method] for method in METHOD_LABELS if method in results_by_method), None)\n    if first_result is None:\n        return\n\n    layer_output_history = first_result["history"].get("layer_output_norms_raw", [])\n    if len(layer_output_history) == 0:\n        return\n\n    num_layers = len(layer_output_history[0])\n    fig, axes = plt.subplots(num_layers, 1, figsize=(10, 3.4 * num_layers), sharex=True, squeeze=False)\n    axes = axes.flatten()\n\n    for layer_idx in range(num_layers):\n        layer_label = "Input layer" if layer_idx == 0 else f"Hidden layer {layer_idx} output"\n        axis = axes[layer_idx]\n        for method in METHOD_LABELS:\n            if method not in results_by_method:\n                continue\n            history = results_by_method[method]["history"]\n            values = [epoch_values[layer_idx] for epoch_values in history.get("layer_output_norms_raw", [])]\n            if len(values) == 0:\n                continue\n            axis.plot(history["epoch"], values, label=METHOD_LABELS[method], color=METHOD_COLORS[method])\n        axis.set_title(f"{layer_label} norm")\n        axis.set_ylabel("Average norm")\n        axis.legend(ncol=2)\n\n    axes[-1].set_xlabel("Epoch")\n    fig.tight_layout()\n    plt.show()\n\n\ndef layer_parameter_slices(model):\n    layer_slices = []\n    start = 0\n    for layer_idx, layer in enumerate(model.layers):\n        layer_parameter_count = layer.weight.numel() + layer.bias.numel()\n        layer_slices.append((layer_idx, slice(start, start + layer_parameter_count), f"Layer {layer_idx + 1}"))\n        start += layer_parameter_count\n    return layer_slices\n\n\ndef analyze_frozen_backprop_estimators(\n    checkpoint_states,\n    data,\n    dimensions,\n    device,\n    method_sigmas,\n    num_perturbations=50,\n    batch_size=128,\n):\n    analysis_dataset = TensorDataset(data["x_train"], data["y_train"])\n    analysis_loader = DataLoader(\n        analysis_dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=(device.type == "cuda"),\n    )\n\n    rows = []\n    perturbation_methods = list(method_sigmas.keys())\n\n    for checkpoint_epoch, state_dict in checkpoint_states.items():\n        model = make_model(dimensions, require_grad=True, device=device)\n        model.load_state_dict(state_dict)\n        component_specs = [(-1, slice(None), "All layers")] + layer_parameter_slices(model)\n\n        batch_metrics = {\n            method: {\n                layer_index: {\n                    "component": "all" if layer_index == -1 else f"layer_{layer_index + 1}",\n                    "component_label": component_label,\n                    "avg_sample_cosine": [],\n                    "mean_estimate_cosine": [],\n                    "sample_variance": [],\n                    "batch_variance": [],\n                }\n                for layer_index, _, component_label in component_specs\n            }\n            for method in perturbation_methods\n        }\n\n        for xb, yb in analysis_loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            true_update = true_gradient_vector(model, xb, yb)\n\n            for method in perturbation_methods:\n                sigma = method_sigmas[method]\n                estimates = [\n                    perturbation_gradient_estimate_vector(method, model, xb, yb, sigma)\n                    for _ in range(num_perturbations)\n                ]\n                estimate_matrix = torch.stack(estimates, dim=0)\n                mean_estimate = estimate_matrix.mean(dim=0)\n\n                for layer_index, component_slice, _ in component_specs:\n                    component_true_update = true_update[component_slice]\n                    component_estimate_matrix = estimate_matrix[:, component_slice]\n                    component_mean_estimate = mean_estimate[component_slice]\n\n                    avg_sample_cosine = float(\n                        np.mean(\n                            [\n                                cosine_similarity_safe(component_estimate, component_true_update)\n                                for component_estimate in component_estimate_matrix\n                            ]\n                        )\n                    )\n                    mean_estimate_cosine = cosine_similarity_safe(component_mean_estimate, component_true_update)\n                    sample_variance, batch_variance = variance_against_true_gradient(\n                        component_estimate_matrix,\n                        component_true_update,\n                    )\n\n                    metrics = batch_metrics[method][layer_index]\n                    metrics["avg_sample_cosine"].append(avg_sample_cosine)\n                    metrics["mean_estimate_cosine"].append(mean_estimate_cosine)\n                    metrics["sample_variance"].append(sample_variance)\n                    metrics["batch_variance"].append(batch_variance)\n\n        checkpoint_fraction = checkpoint_epoch / max(checkpoint_states.keys())\n        for method in perturbation_methods:\n            for layer_index, _, component_label in component_specs:\n                metrics = batch_metrics[method][layer_index]\n                rows.append({\n                    "checkpoint_epoch": checkpoint_epoch,\n                    "checkpoint_fraction": checkpoint_fraction,\n                    "method": method,\n                    "sigma": method_sigmas[method],\n                    "component": metrics["component"],\n                    "component_label": component_label,\n                    "layer_index": layer_index,\n                    "avg_sample_cosine": float(np.mean(metrics["avg_sample_cosine"])),\n                    "mean_estimate_cosine": float(np.mean(metrics["mean_estimate_cosine"])),\n                    "sample_variance": float(np.mean(metrics["sample_variance"])),\n                    "batch_variance": float(np.mean(metrics["batch_variance"])),\n                })\n\n    return pd.DataFrame(rows)\n\n\ndef plot_frozen_estimator_statistics(stats_df):\n    metrics = [\n        ("avg_sample_cosine", "Average Sample Cosine", "Cosine"),\n        ("mean_estimate_cosine", "Cosine of Mean Estimator", "Cosine"),\n        ("sample_variance", "Per-Sample Variance to True Gradient", "Mean squared distance per parameter"),\n        ("batch_variance", "Batch-Average Variance to True Gradient", "Mean squared distance per parameter"),\n    ]\n    perturbation_methods = [\n        method for method in METHOD_LABELS\n        if method in {"np", "np_fan_in", "np_fixed", "wp"} and method in set(stats_df["method"])\n    ]\n\n    overall_df = stats_df[stats_df["layer_index"] == -1] if "layer_index" in stats_df.columns else stats_df\n    fig, axes = plt.subplots(1, len(metrics), figsize=(24, 4), sharex=True)\n\n    for method in perturbation_methods:\n        method_df = overall_df[overall_df["method"] == method].sort_values("checkpoint_epoch")\n        if len(method_df) == 0:\n            continue\n        x = method_df["checkpoint_epoch"].to_numpy()\n        for axis, (column, title, ylabel) in zip(axes, metrics):\n            axis.plot(x, method_df[column].to_numpy(), marker="o", color=METHOD_COLORS[method], label=METHOD_LABELS[method])\n            axis.set_title(f"All layers | {title}")\n            axis.set_xlabel("Checkpoint epoch")\n            axis.set_ylabel(ylabel)\n            axis.legend()\n\n    fig.tight_layout()\n    plt.show()\n\n    if "layer_index" not in stats_df.columns:\n        return\n\n    layer_indices = sorted(layer_index for layer_index in stats_df["layer_index"].unique() if layer_index >= 0)\n    if len(layer_indices) == 0:\n        return\n\n    fig, axes = plt.subplots(\n        len(layer_indices),\n        len(metrics),\n        figsize=(6 * len(metrics), 3.8 * len(layer_indices)),\n        sharex=True,\n        squeeze=False,\n    )\n\n    for row, layer_index in enumerate(layer_indices):\n        layer_df = stats_df[stats_df["layer_index"] == layer_index]\n        component_label = layer_df["component_label"].iloc[0]\n\n        for col, (column, title, ylabel) in enumerate(metrics):\n            axis = axes[row, col]\n            for method in perturbation_methods:\n                method_df = layer_df[layer_df["method"] == method].sort_values("checkpoint_epoch")\n                if len(method_df) == 0:\n                    continue\n                axis.plot(\n                    method_df["checkpoint_epoch"].to_numpy(),\n                    method_df[column].to_numpy(),\n                    marker="o",\n                    color=METHOD_COLORS[method],\n                    label=METHOD_LABELS[method],\n                )\n            axis.set_title(f"{component_label} | {title}")\n            axis.set_xlabel("Checkpoint epoch")\n            axis.set_ylabel(ylabel)\n            if col == len(metrics) - 1:\n                axis.legend(fontsize=8)\n\n    fig.tight_layout()\n    plt.show()\n\n\ndef analyze_frozen_backprop_sigma_grid(\n    checkpoint_states,\n    data,\n    dimensions,\n    device,\n    method_sigma_grid,\n    num_perturbations=50,\n    batch_size=128,\n):\n    analysis_dataset = TensorDataset(data["x_train"], data["y_train"])\n    analysis_loader = DataLoader(\n        analysis_dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=(device.type == "cuda"),\n    )\n\n    rows = []\n    methods = list(method_sigma_grid.keys())\n\n    for checkpoint_epoch, state_dict in checkpoint_states.items():\n        model = make_model(dimensions, require_grad=True, device=device)\n        model.load_state_dict(state_dict)\n\n        batch_metrics = {\n            (method, sigma): {\n                "avg_sample_cosine": [],\n                "mean_estimate_cosine": [],\n                "sample_variance": [],\n                "batch_variance": [],\n            }\n            for method in methods\n            for sigma in method_sigma_grid[method]\n        }\n\n        for batch in analysis_loader:\n            if len(batch) == 3:\n                xb, yb, _ = batch\n            else:\n                xb, yb = batch\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            true_update = true_gradient_vector(model, xb, yb)\n\n            for method in methods:\n                for sigma in method_sigma_grid[method]:\n                    estimates = [\n                        perturbation_gradient_estimate_vector(method, model, xb, yb, sigma)\n                        for _ in range(num_perturbations)\n                    ]\n                    estimate_matrix = torch.stack(estimates, dim=0)\n                    mean_estimate = estimate_matrix.mean(dim=0)\n\n                    avg_sample_cosine = float(\n                        sum(cosine_similarity_safe(estimate, true_update) for estimate in estimates) / num_perturbations\n                    )\n                    mean_estimate_cosine = cosine_similarity_safe(mean_estimate, true_update)\n                    sample_variance, batch_variance = variance_against_true_gradient(estimate_matrix, true_update)\n\n                    metrics = batch_metrics[(method, sigma)]\n                    metrics["avg_sample_cosine"].append(avg_sample_cosine)\n                    metrics["mean_estimate_cosine"].append(mean_estimate_cosine)\n                    metrics["sample_variance"].append(sample_variance)\n                    metrics["batch_variance"].append(batch_variance)\n\n        checkpoint_fraction = checkpoint_epoch / max(checkpoint_states.keys())\n        for method in methods:\n            for sigma in method_sigma_grid[method]:\n                metrics = batch_metrics[(method, sigma)]\n                rows.append({\n                    "checkpoint_epoch": checkpoint_epoch,\n                    "checkpoint_fraction": checkpoint_fraction,\n                    "method": method,\n                    "sigma": float(sigma),\n                    "avg_sample_cosine": float(np.mean(metrics["avg_sample_cosine"])),\n                    "mean_estimate_cosine": float(np.mean(metrics["mean_estimate_cosine"])),\n                    "sample_variance": float(np.mean(metrics["sample_variance"])),\n                    "batch_variance": float(np.mean(metrics["batch_variance"])),\n                })\n\n    return pd.DataFrame(rows)\n\n\ndef plot_frozen_sigma_search_results(stats_df):\n    metrics = [\n        ("avg_sample_cosine", "Average Sample Cosine", "Cosine"),\n        ("mean_estimate_cosine", "Cosine of Mean Estimator", "Cosine"),\n        ("sample_variance", "Per-Sample Variance to True Gradient", "Mean squared distance per parameter"),\n        ("batch_variance", "Batch-Average Variance to True Gradient", "Mean squared distance per parameter"),\n    ]\n    checkpoint_epochs = sorted(stats_df["checkpoint_epoch"].unique())\n\n    for method in [method for method in METHOD_LABELS if method in set(stats_df["method"])] :\n        method_df = stats_df[stats_df["method"] == method]\n        fig, axes = plt.subplots(\n            len(checkpoint_epochs),\n            len(metrics),\n            figsize=(5 * len(metrics), 3.8 * len(checkpoint_epochs)),\n            squeeze=False,\n        )\n\n        for row, checkpoint_epoch in enumerate(checkpoint_epochs):\n            subset = method_df[method_df["checkpoint_epoch"] == checkpoint_epoch].sort_values("sigma")\n            sigma_labels = [f"{sigma:.4g}" for sigma in subset["sigma"].to_numpy()]\n\n            for col, (column, title, ylabel) in enumerate(metrics):\n                axis = axes[row, col]\n                axis.bar(sigma_labels, subset[column].to_numpy(), color=METHOD_COLORS[method])\n                axis.set_title(f"epoch {checkpoint_epoch} | {title}")\n                axis.set_xlabel("sigma")\n                axis.set_ylabel(ylabel)\n                axis.tick_params(axis="x", rotation=45)\n\n        fig.suptitle(f"{METHOD_LABELS[method]} sigma search", y=1.02)\n        fig.tight_layout()\n        plt.show()\n\n\ndef build_grid(space):\n    keys = list(space.keys())\n    values = [space[key] for key in keys]\n    return [dict(zip(keys, combo)) for combo in itertools.product(*values)]\n\n\ndef run_sweep(search_spaces, data, dimensions, sweep_epochs, seeds, device, first_layer_weight_scale=1.0):\n    results = []\n    flat_records = []\n    total_runs = sum(len(configs) * len(seeds) for configs in search_spaces.values())\n    run_idx = 0\n\n    print(f"Starting sweep with total_runs={total_runs}, device={device}")\n    for method in METHODS:\n        configs = search_spaces[method]\n        for config in configs:\n            for seed in seeds:\n                run_idx += 1\n                sigma_str = f", sigma={config[\'sigma\']:.4g}" if "sigma" in config else ""\n                print()\n                print(f"=== Run {run_idx}/{total_runs} | {method} | lr={config[\'lr\']:.4g}{sigma_str} | seed={seed} ===")\n                result = train_one_run(\n                    method=method,\n                    run_config=config,\n                    data=data,\n                    dimensions=dimensions,\n                    epochs=sweep_epochs,\n                    seed=seed,\n                    device=device,\n                    print_every_epoch=1,\n                    first_layer_weight_scale=first_layer_weight_scale,\n                )\n                results.append(result)\n                flat_record = {k: v for k, v in result.items() if k not in {"history", "state_dict"}}\n                flat_record["seed"] = seed\n                flat_records.append(flat_record)\n\n    df = pd.DataFrame(flat_records)\n    if len(df) > 0:\n        df = df.sort_values(\n            ["method", "final_test_loss", "best_test_loss", "final_test_r2"],\n            ascending=[True, True, True, False],\n        ).reset_index(drop=True)\n    return results, df\n\n\ndef summarize_top_configs(df, top_k=5):\n    if len(df) == 0:\n        return df\n    frames = []\n    for method, group in df.groupby("method"):\n        frames.append(group.head(top_k))\n    return pd.concat(frames, ignore_index=True)\n\n\ndef select_best_configs(df):\n    rows = []\n    good = df[df["diverged"] == False]\n    for method, group in good.groupby("method"):\n        best = group.sort_values(\n            ["final_test_loss", "best_test_loss", "final_test_r2"],\n            ascending=[True, True, False],\n        ).iloc[0]\n        rows.append(best)\n    return pd.DataFrame(rows).reset_index(drop=True)\n\n\ndef run_best_config_comparison(best_df, data, dimensions, final_epochs, seed, device, first_layer_weight_scale=1.0):\n    comparison_results = {}\n    for row in best_df.itertuples(index=False):\n        config = {"lr": float(row.lr)}\n        if not pd.isna(row.sigma):\n            config["sigma"] = float(row.sigma)\n        print()\n        print(\n            f"### Final run | {row.method} | lr={config[\'lr\']:.4g}" +\n            (f", sigma={config[\'sigma\']:.4g}" if \'sigma\' in config else "")\n        )\n        comparison_results[row.method] = train_one_run(\n            method=row.method,\n            run_config=config,\n            data=data,\n            dimensions=dimensions,\n            epochs=final_epochs,\n            seed=seed,\n            device=device,\n            print_every_epoch=1,\n            first_layer_weight_scale=first_layer_weight_scale,\n        )\n    return comparison_results\n\n\ndef results_table(results_by_method):\n    rows = []\n    for method in METHOD_LABELS:\n        if method not in results_by_method:\n            continue\n        result = results_by_method[method]\n        rows.append(\n            {\n                "method": method,\n                "final_train_loss": result["final_train_loss"],\n                "final_train_r2": result["final_train_r2"],\n                "final_test_loss": result["final_test_loss"],\n                "final_test_r2": result["final_test_r2"],\n                "best_test_loss": result["best_test_loss"],\n                "best_test_r2": result["best_test_r2"],\n                "diverged": result["diverged"],\n                "duration_sec": result["duration_sec"],\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef plot_training_histories(results_by_method):\n    fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)\n    for method in METHOD_LABELS:\n        if method not in results_by_method:\n            continue\n        result = results_by_method[method]\n        history = result["history"]\n        axes[0].plot(history["epoch"], history["train_loss"], label=f"{METHOD_LABELS[method]} train", color=METHOD_COLORS[method])\n        axes[0].plot(history["epoch"], history["test_loss"], linestyle="--", label=f"{METHOD_LABELS[method]} test", color=METHOD_COLORS[method])\n        axes[1].plot(history["epoch"], history["train_r2"], label=f"{METHOD_LABELS[method]} train", color=METHOD_COLORS[method])\n        axes[1].plot(history["epoch"], history["test_r2"], linestyle="--", label=f"{METHOD_LABELS[method]} test", color=METHOD_COLORS[method])\n        axes[2].plot(history["epoch"], history["weight_norm"], label=METHOD_LABELS[method], color=METHOD_COLORS[method])\n    axes[0].set_title("Loss")\n    axes[0].set_ylabel("MSE")\n    axes[0].legend(ncol=2)\n    axes[1].set_title("R^2")\n    axes[1].set_ylabel("R^2")\n    axes[1].legend(ncol=2)\n    axes[2].set_title("Weight Norm")\n    axes[2].set_ylabel("L2 norm")\n    axes[2].set_xlabel("Epoch")\n    axes[2].legend(ncol=2)\n    fig.tight_layout()\n    plt.show()\n\n\ndef plot_layer_output_norm_histories(results_by_method):\n    first_result = next((results_by_method[method] for method in METHOD_LABELS if method in results_by_method), None)\n    if first_result is None:\n        return\n\n    layer_output_history = first_result["history"].get("layer_output_norms", [])\n    if len(layer_output_history) == 0:\n        return\n\n    num_layers = len(layer_output_history[0])\n    fig, axes = plt.subplots(num_layers, 1, figsize=(10, 3.4 * num_layers), sharex=True, squeeze=False)\n    axes = axes.flatten()\n\n    for layer_idx in range(num_layers):\n        fan_in = int(first_result["state_dict"][f"layers.{layer_idx}.weight"].shape[1])\n        layer_label = "Input layer" if layer_idx == 0 else f"Hidden layer {layer_idx} output"\n        axis = axes[layer_idx]\n        for method in METHOD_LABELS:\n            if method not in results_by_method:\n                continue\n            history = results_by_method[method]["history"]\n            values = [epoch_values[layer_idx] for epoch_values in history.get("layer_output_norms", [])]\n            if len(values) == 0:\n                continue\n            axis.plot(history["epoch"], values, label=METHOD_LABELS[method], color=METHOD_COLORS[method])\n        axis.set_title(f"{layer_label} norm / sqrt({fan_in} + 1)")\n        axis.set_ylabel("Average norm")\n        axis.legend(ncol=2)\n\n    axes[-1].set_xlabel("Epoch")\n    fig.tight_layout()\n    plt.show()\n\n\ndef synthetic_target(latent):\n    input_dim = latent.shape[1]\n    target = 0.6 * torch.sin(math.pi * latent).sum(dim=1, keepdim=True)\n    target = target + 0.3 * latent.pow(2).sum(dim=1, keepdim=True)\n    target = target + 0.2 * (latent[:, 0:1] * latent[:, 1:2])\n    return target / math.sqrt(input_dim)\n\n\ndef load_synthetic_vector_regression_data(\n    input_dim=8,\n    input_scale=1.0,\n    n_train=512,\n    n_test=512,\n    noise_std=0.1,\n    train_eval_limit=None,\n    batch_size=64,\n    eval_batch_size=512,\n    seed=0,\n    num_workers=0,\n):\n    generator = torch.Generator().manual_seed(seed)\n    latent_train = torch.rand(n_train, input_dim, generator=generator) * 2.0 - 1.0\n    x_train = input_scale * latent_train\n    y_train = synthetic_target(latent_train) + noise_std * torch.randn(n_train, 1, generator=generator)\n\n    latent_test = torch.rand(n_test, input_dim, generator=generator) * 2.0 - 1.0\n    x_test = input_scale * latent_test\n    y_test = synthetic_target(latent_test)\n\n    train_dataset = TensorDataset(x_train, y_train)\n    test_dataset = TensorDataset(x_test, y_test)\n    train_eval_dataset = _maybe_subset_dataset(train_dataset, train_eval_limit, seed + 1)\n\n    pin_memory = device.type == "cuda"\n    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)\n    train_eval_loader = DataLoader(train_eval_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)\n    test_loader = DataLoader(test_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)\n\n    return {\n        "train_loader": train_loader,\n        "train_eval_loader": train_eval_loader,\n        "test_loader": test_loader,\n        "train_size": len(train_dataset),\n        "train_eval_size": len(train_eval_dataset),\n        "test_size": len(test_dataset),\n        "x_train": x_train,\n        "y_train": y_train,\n        "x_test": x_test,\n        "y_test": y_test,\n        "latent_train": latent_train,\n        "latent_test": latent_test,\n        "input_scale": input_scale,\n    }\n\n\ndef plot_prediction_diagnostics(results_by_method, data, dimensions, device):\n    x_train = data["x_train"]\n    y_train = data["y_train"]\n    x_test = data["x_test"]\n    y_test = data["y_test"]\n\n    plt.figure(figsize=(10, 5))\n    plt.plot(x_test.squeeze().numpy(), y_test.squeeze().numpy(), color="black", linewidth=2, label="True signal")\n    plt.scatter(x_train.squeeze().numpy(), y_train.squeeze().numpy(), color="0.8", s=12, alpha=0.5, label="Train samples")\n\n    for method in METHOD_LABELS:\n        if method not in results_by_method:\n            continue\n        model = make_model(dimensions, require_grad=(method == "bp"), device=device)\n        model.load_state_dict(results_by_method[method]["state_dict"])\n        prediction = predict_tensor(model, x_test, device)\n        plt.plot(x_test.squeeze().numpy(), prediction.squeeze().numpy(), color=METHOD_COLORS[method], label=METHOD_LABELS[method])\n\n    plt.title("Sinus Prediction Curves")\n    plt.xlabel("x")\n    plt.ylabel("y")\n    plt.legend()\n    plt.tight_layout()\n    plt.show()\n\n\n',
    'california_housing': 'import itertools\nimport math\nimport random\nimport time\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom IPython.display import display\nfrom torch.utils.data import DataLoader, TensorDataset, Subset\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nif hasattr(torch, "set_float32_matmul_precision"):\n    torch.set_float32_matmul_precision("high")\n\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\nset_seed(0)\nprint(f"device={device}")\nif device.type == "cuda":\n    print(torch.cuda.get_device_name(0))\n\nfrom sklearn.datasets import fetch_california_housing\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\n\n\nMETHODS = ["bp", "np", "np_fan_in", "np_fixed", "wp"]\nMETHOD_LABELS = {\n    "bp": "BP",\n    "np": "IS-NP",\n    "np_fan_in": "Fan-in NP",\n    "np_fixed": "Vanilla NP",\n    "wp": "WP",\n}\nMETHOD_COLORS = {\n    "bp": "C0",\n    "np": "C1",\n    "np_fan_in": "C4",\n    "np_fixed": "C3",\n    "wp": "C2",\n}\n\n\nclass MLP(nn.Module):\n    def __init__(self, dimensions, activation=torch.sigmoid, output_activation=None, require_grad=False):\n        super().__init__()\n        self.layers = nn.ModuleList(\n            [nn.Linear(dimensions[i], dimensions[i + 1], bias=True) for i in range(len(dimensions) - 1)]\n        )\n        self.activation = activation\n        self.output_activation = output_activation\n        if not require_grad:\n            for p in self.parameters():\n                p.requires_grad_(False)\n\n    def forward(self, x):\n        final_layer = len(self.layers) - 1\n        h = x\n        for i, layer in enumerate(self.layers):\n            u = layer(h)\n            if i == final_layer:\n                h = self.output_activation(u) if self.output_activation else u\n            else:\n                h = self.activation(u)\n        return h\n\n    def forward_weight_perturb(self, x, sigma):\n        batch_size = x.shape[0]\n        xs = [x]\n        ys = []\n        noises = []\n        h = x\n        for i, layer in enumerate(self.layers):\n            w = layer.weight\n            b = layer.bias\n            eps_w = torch.randn(batch_size, *w.shape, device=w.device, dtype=w.dtype) * sigma\n            eps_b = torch.randn(batch_size, *b.shape, device=b.device, dtype=b.dtype) * sigma\n            w_used = w.unsqueeze(0) + eps_w\n            b_used = b.unsqueeze(0) + eps_b\n            u = torch.bmm(h.unsqueeze(1), w_used.transpose(1, 2)).squeeze(1) + b_used\n            final_layer = len(self.layers) - 1\n            if i == final_layer:\n                h = self.output_activation(u) if self.output_activation else u\n                ys.append(h)\n            else:\n                h = self.activation(u)\n                ys.append(h)\n                xs.append(h)\n            noises.append((eps_w, eps_b))\n        return ys, xs, noises\n\n    def forward_node_perturb(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            x_in = a_noisy\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            noise_scale = sigma * torch.sqrt(1.0 + x_in.pow(2).sum(dim=1, keepdim=True))\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n    def forward_node_perturb_fan_in_scaled(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            fan_in_with_bias = layer.in_features + 1\n            noise_scale_value = sigma * math.sqrt(fan_in_with_bias)\n            noise_scale = torch.full_like(z_noisy, noise_scale_value)\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n    def forward_node_perturb_fixed_sigma(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            noise_scale = torch.full_like(z_noisy, sigma)\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n\ndef _mean_loss_per_sample(prediction, target):\n    loss = F.mse_loss(prediction, target, reduction="none")\n    if loss.dim() > 1:\n        loss = loss.mean(dim=1)\n    return loss.view(-1)\n\n\ndef _centered_reward_signal(loss_per_sample):\n    reward = -loss_per_sample\n    return reward - reward.mean()\n\n\ndef _flatten_parameter_tensors(weight_tensors, bias_tensors):\n    pieces = []\n    for weight_tensor, bias_tensor in zip(weight_tensors, bias_tensors):\n        pieces.append(weight_tensor.reshape(-1))\n        pieces.append(bias_tensor.reshape(-1))\n    return torch.cat(pieces)\n\n\ndef node_perturbation_step(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef node_perturbation_step_fixed_sigma(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fixed_sigma(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef node_perturbation_step_fan_in_scaled(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fan_in_scaled(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef weight_perturb_step(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    layer_outputs, _, noises = model.forward_weight_perturb(X, sigma)\n    prediction_noisy = layer_outputs[-1]\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    noise_scale = sigma ** 2 + 1e-12\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, (weight_noise, bias_noise) in zip(model.layers, noises):\n            scaled_weight_noise = scalar_signal.view(-1, 1, 1) * weight_noise / noise_scale\n            scaled_bias_noise = scalar_signal.view(-1, 1) * bias_noise / noise_scale\n            raw_weight_update = scaled_weight_noise.mean(dim=0)\n            raw_bias_update = scaled_bias_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef backprop_step(model, X, target, optimizer, loss_fn=F.mse_loss, return_unscaled_parameter_update_vector=False):\n    model.train()\n    for p in model.parameters():\n        p.requires_grad_(True)\n    optimizer.zero_grad()\n    y_pred = model(X)\n    loss = loss_fn(y_pred, target, reduction="mean")\n    loss.backward()\n    weight_grads = [layer.weight.grad.detach().clone() for layer in model.layers]\n    bias_grads = [layer.bias.grad.detach().clone() for layer in model.layers]\n    optimizer.step()\n    if return_unscaled_parameter_update_vector:\n        return loss.item(), -_flatten_parameter_tensors(weight_grads, bias_grads)\n    return loss.item()\n\n\ndef model_weight_norm(model):\n    total = torch.tensor(0.0, device=next(model.parameters()).device)\n    for layer in model.layers:\n        total = total + layer.weight.detach().pow(2).sum()\n    return float(torch.sqrt(total))\n\n\ndef mean_normalized_layer_output_norms(model, loader, device):\n    model.eval()\n    summed_norms = [0.0 for _ in model.layers]\n    total_examples = 0\n\n    with torch.no_grad():\n        for batch in loader:\n            if len(batch) == 3:\n                xb, _, _ = batch\n            else:\n                xb, _ = batch\n            xb = xb.to(device, non_blocking=True)\n            h = xb\n            batch_size = xb.size(0)\n\n            for layer_idx, layer in enumerate(model.layers):\n                h_for_norm = h if h.dim() > 1 else h.unsqueeze(1)\n                normalized_norm = torch.norm(h_for_norm, dim=1) / math.sqrt(h_for_norm.shape[1] + 1)\n                summed_norms[layer_idx] += normalized_norm.sum().item()\n                u = layer(h)\n                if layer_idx == len(model.layers) - 1:\n                    h = model.output_activation(u) if model.output_activation else u\n                else:\n                    h = model.activation(u)\n\n            total_examples += batch_size\n\n    return [value / max(total_examples, 1) for value in summed_norms]\n\n\ndef _random_subset_indices(n, limit, seed):\n    if limit is None or limit >= n:\n        return torch.arange(n)\n    g = torch.Generator().manual_seed(seed)\n    return torch.randperm(n, generator=g)[:limit]\n\n\ndef _maybe_subset_dataset(dataset, limit, seed):\n    if limit is None or limit >= len(dataset):\n        return dataset\n    indices = _random_subset_indices(len(dataset), limit, seed)\n    return Subset(dataset, indices.tolist())\n\n\ndef regression_r2_score(y_true, y_pred):\n    ss_res = torch.sum((y_true - y_pred) ** 2)\n    target_mean = torch.mean(y_true, dim=0, keepdim=True)\n    ss_tot = torch.sum((y_true - target_mean) ** 2)\n    if float(ss_tot) < 1e-12:\n        return 0.0\n    return float(1.0 - ss_res / (ss_tot + 1e-12))\n\n\ndef evaluate_model(model, loader, device):\n    model.eval()\n    total_loss = 0.0\n    total_examples = 0\n    predictions = []\n    targets = []\n    with torch.no_grad():\n        for xb, yb in loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            pred = model(xb)\n            loss = F.mse_loss(pred, yb, reduction="mean")\n            total_loss += loss.item() * xb.size(0)\n            total_examples += xb.size(0)\n            predictions.append(pred.detach().cpu())\n            targets.append(yb.detach().cpu())\n    mean_loss = total_loss / max(total_examples, 1)\n    y_pred = torch.cat(predictions, dim=0)\n    y_true = torch.cat(targets, dim=0)\n    r2 = regression_r2_score(y_true, y_pred)\n    return mean_loss, r2\n\n\ndef predict_tensor(model, x, device):\n    model.eval()\n    with torch.no_grad():\n        return model(x.to(device)).detach().cpu()\n\n\ndef make_model(dimensions, require_grad, device):\n    return MLP(dimensions, activation=torch.sigmoid, require_grad=require_grad).to(device)\n\n\ndef train_one_run(\n    method,\n    run_config,\n    data,\n    dimensions,\n    epochs,\n    seed,\n    device,\n    print_every_epoch=1,\n    divergence_loss_threshold=5.0,\n):\n    set_seed(seed)\n\n    base_model = make_model(dimensions, require_grad=True, device=device)\n    base_state = {name: tensor.detach().clone() for name, tensor in base_model.state_dict().items()}\n\n    require_grad = method == "bp"\n    model = make_model(dimensions, require_grad=require_grad, device=device)\n    model.load_state_dict(base_state)\n\n    optimizer = None\n    if method == "bp":\n        optimizer = torch.optim.SGD(model.parameters(), lr=run_config["lr"])\n\n    history = {\n        "epoch": [],\n        "train_loss": [],\n        "train_r2": [],\n        "test_loss": [],\n        "test_r2": [],\n        "weight_norm": [],\n        "layer_output_norms": [],\n    }\n\n    diverged = False\n    best_test_loss = float("inf")\n    best_test_r2 = -float("inf")\n    start_time = time.time()\n\n    train_loader = data["train_loader"]\n    train_eval_loader = data["train_eval_loader"]\n    test_loader = data["test_loader"]\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        for xb, yb in train_loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n\n            if method == "bp":\n                backprop_step(model, xb, yb, optimizer=optimizer)\n            elif method == "np":\n                node_perturbation_step(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "np_fan_in":\n                node_perturbation_step_fan_in_scaled(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "np_fixed":\n                node_perturbation_step_fixed_sigma(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "wp":\n                weight_perturb_step(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            else:\n                raise ValueError(f"Unknown method: {method}")\n\n        train_loss, train_r2 = evaluate_model(model, train_eval_loader, device)\n        test_loss, test_r2 = evaluate_model(model, test_loader, device)\n\n        history["epoch"].append(epoch)\n        history["train_loss"].append(train_loss)\n        history["train_r2"].append(train_r2)\n        history["test_loss"].append(test_loss)\n        history["test_r2"].append(test_r2)\n        history["weight_norm"].append(model_weight_norm(model))\n        history["layer_output_norms"].append(mean_normalized_layer_output_norms(model, train_eval_loader, device))\n\n        best_test_loss = min(best_test_loss, test_loss)\n        best_test_r2 = max(best_test_r2, test_r2)\n\n        if (not math.isfinite(train_loss)) or (not math.isfinite(test_loss)) or test_loss > divergence_loss_threshold:\n            diverged = True\n            print(\n                f"    diverged at epoch {epoch}/{epochs} | "\n                f"train_loss={train_loss:.4f}, test_loss={test_loss:.4f}, test_r2={test_r2:.4f}"\n            )\n            break\n\n        if epoch % print_every_epoch == 0 or epoch == 1 or epoch == epochs:\n            sigma_str = f", sigma={run_config[\'sigma\']:.4g}" if "sigma" in run_config else ""\n            print(\n                f"    epoch {epoch:2d}/{epochs} | {method} | lr={run_config[\'lr\']:.4g}{sigma_str} | "\n                f"train_loss={train_loss:.4f}, train_r2={train_r2:.4f}, "\n                f"test_loss={test_loss:.4f}, test_r2={test_r2:.4f}"\n            )\n\n    duration_sec = time.time() - start_time\n    state_dict_cpu = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n    result = {\n        "method": method,\n        "lr": float(run_config["lr"]),\n        "sigma": float(run_config["sigma"]) if "sigma" in run_config else np.nan,\n        "epochs_completed": len(history["epoch"]),\n        "final_train_loss": history["train_loss"][-1] if history["train_loss"] else np.nan,\n        "final_train_r2": history["train_r2"][-1] if history["train_r2"] else np.nan,\n        "final_test_loss": history["test_loss"][-1] if history["test_loss"] else np.nan,\n        "final_test_r2": history["test_r2"][-1] if history["test_r2"] else np.nan,\n        "best_test_loss": best_test_loss,\n        "best_test_r2": best_test_r2,\n        "diverged": diverged,\n        "duration_sec": duration_sec,\n        "history": history,\n        "state_dict": state_dict_cpu,\n    }\n    del model\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return result\n\n\ndef true_gradient_vector(model, xb, yb):\n    requires_grad_state = [parameter.requires_grad for parameter in model.parameters()]\n    for parameter in model.parameters():\n        parameter.requires_grad_(True)\n    model.zero_grad(set_to_none=True)\n    prediction = model(xb)\n    loss = F.mse_loss(prediction, yb, reduction="mean")\n    loss.backward()\n    weight_grads = [layer.weight.grad.detach().clone() for layer in model.layers]\n    bias_grads = [layer.bias.grad.detach().clone() for layer in model.layers]\n    flat_grad = _flatten_parameter_tensors(weight_grads, bias_grads)\n    model.zero_grad(set_to_none=True)\n    for parameter, old_value in zip(model.parameters(), requires_grad_state):\n        parameter.requires_grad_(old_value)\n    return -flat_grad\n\n\ndef cosine_similarity_safe(a, b, eps=1e-12):\n    a_norm = torch.norm(a)\n    b_norm = torch.norm(b)\n    if a_norm.item() < eps or b_norm.item() < eps:\n        return 0.0\n    return float(torch.dot(a, b) / (a_norm * b_norm + eps))\n\n\ndef node_perturbation_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef node_perturbation_fan_in_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fan_in_scaled(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef node_perturbation_fixed_sigma_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fixed_sigma(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef weight_perturbation_gradient_estimate_vector(model, xb, yb, sigma):\n    layer_outputs, _, noises = model.forward_weight_perturb(xb, sigma)\n    prediction_noisy = layer_outputs[-1]\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    noise_scale = sigma ** 2 + 1e-12\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for weight_noise, bias_noise in noises:\n        scaled_weight_noise = scalar_signal.view(-1, 1, 1) * weight_noise / noise_scale\n        scaled_bias_noise = scalar_signal.view(-1, 1) * bias_noise / noise_scale\n        raw_weight_updates.append(scaled_weight_noise.mean(dim=0))\n        raw_bias_updates.append(scaled_bias_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef perturbation_gradient_estimate_vector(method, model, xb, yb, sigma):\n    if method == "np":\n        return node_perturbation_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "np_fan_in":\n        return node_perturbation_fan_in_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "np_fixed":\n        return node_perturbation_fixed_sigma_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "wp":\n        return weight_perturbation_gradient_estimate_vector(model, xb, yb, sigma)\n    raise ValueError(f"Unknown perturbation method: {method}")\n\n\ndef variance_against_true_gradient(estimate_matrix, true_update):\n    num_samples = estimate_matrix.shape[0]\n    if num_samples == 0:\n        return 0.0, 0.0\n\n    squared_errors = (estimate_matrix - true_update.unsqueeze(0)).pow(2)\n    per_sample_variance = float(squared_errors.mean(dim=1).mean())\n    mean_estimate = estimate_matrix.mean(dim=0)\n    batch_variance = float((mean_estimate - true_update).pow(2).mean())\n    return per_sample_variance, batch_variance\n\n\ndef train_backprop_checkpoint_states(data, dimensions, epochs, seed, device, lr, checkpoint_epochs, print_every_epoch=100):\n    checkpoint_epochs = sorted(set(int(epoch) for epoch in checkpoint_epochs))\n    set_seed(seed)\n    model = make_model(dimensions, require_grad=True, device=device)\n    optimizer = torch.optim.SGD(model.parameters(), lr=lr)\n    checkpoint_states = {}\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        for xb, yb in data["train_loader"]:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            backprop_step(model, xb, yb, optimizer=optimizer)\n\n        if epoch in checkpoint_epochs:\n            checkpoint_states[epoch] = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n\n        if epoch % print_every_epoch == 0 or epoch == 1 or epoch == epochs or epoch in checkpoint_epochs:\n            train_loss, train_r2 = evaluate_model(model, data["train_eval_loader"], device)\n            test_loss, test_r2 = evaluate_model(model, data["test_loader"], device)\n            print(\n                f"    bp checkpoint training epoch {epoch:4d}/{epochs} | "\n                f"train_loss={train_loss:.4f}, train_r2={train_r2:.4f}, "\n                f"test_loss={test_loss:.4f}, test_r2={test_r2:.4f}"\n            )\n\n    return checkpoint_states\n\n\ndef layer_parameter_slices(model):\n    layer_slices = []\n    start = 0\n    for layer_idx, layer in enumerate(model.layers):\n        layer_parameter_count = layer.weight.numel() + layer.bias.numel()\n        layer_slices.append((layer_idx, slice(start, start + layer_parameter_count), f"Layer {layer_idx + 1}"))\n        start += layer_parameter_count\n    return layer_slices\n\n\ndef analyze_frozen_backprop_estimators(\n    checkpoint_states,\n    data,\n    dimensions,\n    device,\n    method_sigmas,\n    num_perturbations=50,\n    batch_size=128,\n):\n    analysis_dataset = TensorDataset(data["x_train"], data["y_train"])\n    analysis_loader = DataLoader(\n        analysis_dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=(device.type == "cuda"),\n    )\n\n    rows = []\n    perturbation_methods = list(method_sigmas.keys())\n\n    for checkpoint_epoch, state_dict in checkpoint_states.items():\n        model = make_model(dimensions, require_grad=True, device=device)\n        model.load_state_dict(state_dict)\n        component_specs = [(-1, slice(None), "All layers")] + layer_parameter_slices(model)\n\n        batch_metrics = {\n            method: {\n                layer_index: {\n                    "component": "all" if layer_index == -1 else f"layer_{layer_index + 1}",\n                    "component_label": component_label,\n                    "avg_sample_cosine": [],\n                    "mean_estimate_cosine": [],\n                    "sample_variance": [],\n                    "batch_variance": [],\n                }\n                for layer_index, _, component_label in component_specs\n            }\n            for method in perturbation_methods\n        }\n\n        for xb, yb in analysis_loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            true_update = true_gradient_vector(model, xb, yb)\n\n            for method in perturbation_methods:\n                sigma = method_sigmas[method]\n                estimates = [\n                    perturbation_gradient_estimate_vector(method, model, xb, yb, sigma)\n                    for _ in range(num_perturbations)\n                ]\n                estimate_matrix = torch.stack(estimates, dim=0)\n                mean_estimate = estimate_matrix.mean(dim=0)\n\n                for layer_index, component_slice, _ in component_specs:\n                    component_true_update = true_update[component_slice]\n                    component_estimate_matrix = estimate_matrix[:, component_slice]\n                    component_mean_estimate = mean_estimate[component_slice]\n\n                    avg_sample_cosine = float(\n                        np.mean(\n                            [\n                                cosine_similarity_safe(component_estimate, component_true_update)\n                                for component_estimate in component_estimate_matrix\n                            ]\n                        )\n                    )\n                    mean_estimate_cosine = cosine_similarity_safe(component_mean_estimate, component_true_update)\n                    sample_variance, batch_variance = variance_against_true_gradient(\n                        component_estimate_matrix,\n                        component_true_update,\n                    )\n\n                    metrics = batch_metrics[method][layer_index]\n                    metrics["avg_sample_cosine"].append(avg_sample_cosine)\n                    metrics["mean_estimate_cosine"].append(mean_estimate_cosine)\n                    metrics["sample_variance"].append(sample_variance)\n                    metrics["batch_variance"].append(batch_variance)\n\n        checkpoint_fraction = checkpoint_epoch / max(checkpoint_states.keys())\n        for method in perturbation_methods:\n            for layer_index, _, component_label in component_specs:\n                metrics = batch_metrics[method][layer_index]\n                rows.append({\n                    "checkpoint_epoch": checkpoint_epoch,\n                    "checkpoint_fraction": checkpoint_fraction,\n                    "method": method,\n                    "sigma": method_sigmas[method],\n                    "component": metrics["component"],\n                    "component_label": component_label,\n                    "layer_index": layer_index,\n                    "avg_sample_cosine": float(np.mean(metrics["avg_sample_cosine"])),\n                    "mean_estimate_cosine": float(np.mean(metrics["mean_estimate_cosine"])),\n                    "sample_variance": float(np.mean(metrics["sample_variance"])),\n                    "batch_variance": float(np.mean(metrics["batch_variance"])),\n                })\n\n    return pd.DataFrame(rows)\n\n\ndef plot_frozen_estimator_statistics(stats_df):\n    metrics = [\n        ("avg_sample_cosine", "Average Sample Cosine", "Cosine"),\n        ("mean_estimate_cosine", "Cosine of Mean Estimator", "Cosine"),\n        ("sample_variance", "Per-Sample Variance to True Gradient", "Mean squared distance per parameter"),\n        ("batch_variance", "Batch-Average Variance to True Gradient", "Mean squared distance per parameter"),\n    ]\n    perturbation_methods = [method for method in METHODS if method in {"np", "np_fan_in", "np_fixed", "wp"}]\n\n    overall_df = stats_df[stats_df["layer_index"] == -1] if "layer_index" in stats_df.columns else stats_df\n    fig, axes = plt.subplots(1, len(metrics), figsize=(24, 4), sharex=True)\n\n    for method in perturbation_methods:\n        method_df = overall_df[overall_df["method"] == method].sort_values("checkpoint_epoch")\n        if len(method_df) == 0:\n            continue\n        x = method_df["checkpoint_epoch"].to_numpy()\n        for axis, (column, title, ylabel) in zip(axes, metrics):\n            axis.plot(x, method_df[column].to_numpy(), marker="o", color=METHOD_COLORS[method], label=METHOD_LABELS[method])\n            axis.set_title(f"All layers | {title}")\n            axis.set_xlabel("Checkpoint epoch")\n            axis.set_ylabel(ylabel)\n            axis.legend()\n\n    fig.tight_layout()\n    plt.show()\n\n    if "layer_index" not in stats_df.columns:\n        return\n\n    layer_indices = sorted(layer_index for layer_index in stats_df["layer_index"].unique() if layer_index >= 0)\n    if len(layer_indices) == 0:\n        return\n\n    fig, axes = plt.subplots(\n        len(layer_indices),\n        len(metrics),\n        figsize=(6 * len(metrics), 3.8 * len(layer_indices)),\n        sharex=True,\n        squeeze=False,\n    )\n\n    for row, layer_index in enumerate(layer_indices):\n        layer_df = stats_df[stats_df["layer_index"] == layer_index]\n        component_label = layer_df["component_label"].iloc[0]\n\n        for col, (column, title, ylabel) in enumerate(metrics):\n            axis = axes[row, col]\n            for method in perturbation_methods:\n                method_df = layer_df[layer_df["method"] == method].sort_values("checkpoint_epoch")\n                if len(method_df) == 0:\n                    continue\n                axis.plot(\n                    method_df["checkpoint_epoch"].to_numpy(),\n                    method_df[column].to_numpy(),\n                    marker="o",\n                    color=METHOD_COLORS[method],\n                    label=METHOD_LABELS[method],\n                )\n            axis.set_title(f"{component_label} | {title}")\n            axis.set_xlabel("Checkpoint epoch")\n            axis.set_ylabel(ylabel)\n            if col == len(metrics) - 1:\n                axis.legend(fontsize=8)\n\n    fig.tight_layout()\n    plt.show()\n\n\ndef analyze_frozen_backprop_sigma_grid(\n    checkpoint_states,\n    data,\n    dimensions,\n    device,\n    method_sigma_grid,\n    num_perturbations=50,\n    batch_size=128,\n):\n    analysis_dataset = TensorDataset(data["x_train"], data["y_train"])\n    analysis_loader = DataLoader(\n        analysis_dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=(device.type == "cuda"),\n    )\n\n    rows = []\n    methods = list(method_sigma_grid.keys())\n\n    for checkpoint_epoch, state_dict in checkpoint_states.items():\n        model = make_model(dimensions, require_grad=True, device=device)\n        model.load_state_dict(state_dict)\n\n        batch_metrics = {\n            (method, sigma): {\n                "avg_sample_cosine": [],\n                "mean_estimate_cosine": [],\n                "sample_variance": [],\n                "batch_variance": [],\n            }\n            for method in methods\n            for sigma in method_sigma_grid[method]\n        }\n\n        for batch in analysis_loader:\n            if len(batch) == 3:\n                xb, yb, _ = batch\n            else:\n                xb, yb = batch\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            true_update = true_gradient_vector(model, xb, yb)\n\n            for method in methods:\n                for sigma in method_sigma_grid[method]:\n                    estimates = [\n                        perturbation_gradient_estimate_vector(method, model, xb, yb, sigma)\n                        for _ in range(num_perturbations)\n                    ]\n                    estimate_matrix = torch.stack(estimates, dim=0)\n                    mean_estimate = estimate_matrix.mean(dim=0)\n\n                    avg_sample_cosine = float(\n                        sum(cosine_similarity_safe(estimate, true_update) for estimate in estimates) / num_perturbations\n                    )\n                    mean_estimate_cosine = cosine_similarity_safe(mean_estimate, true_update)\n                    sample_variance, batch_variance = variance_against_true_gradient(estimate_matrix, true_update)\n\n                    metrics = batch_metrics[(method, sigma)]\n                    metrics["avg_sample_cosine"].append(avg_sample_cosine)\n                    metrics["mean_estimate_cosine"].append(mean_estimate_cosine)\n                    metrics["sample_variance"].append(sample_variance)\n                    metrics["batch_variance"].append(batch_variance)\n\n        checkpoint_fraction = checkpoint_epoch / max(checkpoint_states.keys())\n        for method in methods:\n            for sigma in method_sigma_grid[method]:\n                metrics = batch_metrics[(method, sigma)]\n                rows.append({\n                    "checkpoint_epoch": checkpoint_epoch,\n                    "checkpoint_fraction": checkpoint_fraction,\n                    "method": method,\n                    "sigma": float(sigma),\n                    "avg_sample_cosine": float(np.mean(metrics["avg_sample_cosine"])),\n                    "mean_estimate_cosine": float(np.mean(metrics["mean_estimate_cosine"])),\n                    "sample_variance": float(np.mean(metrics["sample_variance"])),\n                    "batch_variance": float(np.mean(metrics["batch_variance"])),\n                })\n\n    return pd.DataFrame(rows)\n\n\ndef plot_frozen_sigma_search_results(stats_df):\n    metrics = [\n        ("avg_sample_cosine", "Average Sample Cosine", "Cosine"),\n        ("mean_estimate_cosine", "Cosine of Mean Estimator", "Cosine"),\n        ("sample_variance", "Per-Sample Variance to True Gradient", "Mean squared distance per parameter"),\n        ("batch_variance", "Batch-Average Variance to True Gradient", "Mean squared distance per parameter"),\n    ]\n    checkpoint_epochs = sorted(stats_df["checkpoint_epoch"].unique())\n\n    for method in [method for method in METHODS if method in set(stats_df["method"])] :\n        method_df = stats_df[stats_df["method"] == method]\n        fig, axes = plt.subplots(\n            len(checkpoint_epochs),\n            len(metrics),\n            figsize=(5 * len(metrics), 3.8 * len(checkpoint_epochs)),\n            squeeze=False,\n        )\n\n        for row, checkpoint_epoch in enumerate(checkpoint_epochs):\n            subset = method_df[method_df["checkpoint_epoch"] == checkpoint_epoch].sort_values("sigma")\n            sigma_labels = [f"{sigma:.4g}" for sigma in subset["sigma"].to_numpy()]\n\n            for col, (column, title, ylabel) in enumerate(metrics):\n                axis = axes[row, col]\n                axis.bar(sigma_labels, subset[column].to_numpy(), color=METHOD_COLORS[method])\n                axis.set_title(f"epoch {checkpoint_epoch} | {title}")\n                axis.set_xlabel("sigma")\n                axis.set_ylabel(ylabel)\n                axis.tick_params(axis="x", rotation=45)\n\n        fig.suptitle(f"{METHOD_LABELS[method]} sigma search", y=1.02)\n        fig.tight_layout()\n        plt.show()\n\n\ndef build_grid(space):\n    keys = list(space.keys())\n    values = [space[key] for key in keys]\n    return [dict(zip(keys, combo)) for combo in itertools.product(*values)]\n\n\ndef run_sweep(search_spaces, data, dimensions, sweep_epochs, seeds, device):\n    results = []\n    flat_records = []\n    total_runs = sum(len(configs) * len(seeds) for configs in search_spaces.values())\n    run_idx = 0\n\n    print(f"Starting sweep with total_runs={total_runs}, device={device}")\n    for method in METHODS:\n        configs = search_spaces[method]\n        for config in configs:\n            for seed in seeds:\n                run_idx += 1\n                sigma_str = f", sigma={config[\'sigma\']:.4g}" if "sigma" in config else ""\n                print()\n                print(f"=== Run {run_idx}/{total_runs} | {method} | lr={config[\'lr\']:.4g}{sigma_str} | seed={seed} ===")\n                result = train_one_run(\n                    method=method,\n                    run_config=config,\n                    data=data,\n                    dimensions=dimensions,\n                    epochs=sweep_epochs,\n                    seed=seed,\n                    device=device,\n                    print_every_epoch=1,\n                )\n                results.append(result)\n                flat_record = {k: v for k, v in result.items() if k not in {"history", "state_dict"}}\n                flat_record["seed"] = seed\n                flat_records.append(flat_record)\n\n    df = pd.DataFrame(flat_records)\n    if len(df) > 0:\n        df = df.sort_values(\n            ["method", "final_test_loss", "best_test_loss", "final_test_r2"],\n            ascending=[True, True, True, False],\n        ).reset_index(drop=True)\n    return results, df\n\n\ndef summarize_top_configs(df, top_k=5):\n    if len(df) == 0:\n        return df\n    frames = []\n    for method, group in df.groupby("method"):\n        frames.append(group.head(top_k))\n    return pd.concat(frames, ignore_index=True)\n\n\ndef select_best_configs(df):\n    rows = []\n    good = df[df["diverged"] == False]\n    for method, group in good.groupby("method"):\n        best = group.sort_values(\n            ["final_test_loss", "best_test_loss", "final_test_r2"],\n            ascending=[True, True, False],\n        ).iloc[0]\n        rows.append(best)\n    return pd.DataFrame(rows).reset_index(drop=True)\n\n\ndef run_best_config_comparison(best_df, data, dimensions, final_epochs, seed, device):\n    comparison_results = {}\n    for row in best_df.itertuples(index=False):\n        config = {"lr": float(row.lr)}\n        if not pd.isna(row.sigma):\n            config["sigma"] = float(row.sigma)\n        print(\n            f"### Final run | {row.method} | lr={config[\'lr\']:.4g}" +\n            (f", sigma={config[\'sigma\']:.4g}" if \'sigma\' in config else "")\n        )\n        comparison_results[row.method] = train_one_run(\n            method=row.method,\n            run_config=config,\n            data=data,\n            dimensions=dimensions,\n            epochs=final_epochs,\n            seed=seed,\n            device=device,\n            print_every_epoch=1,\n        )\n    return comparison_results\n\n\ndef results_table(results_by_method):\n    rows = []\n    for method in METHODS:\n        if method not in results_by_method:\n            continue\n        result = results_by_method[method]\n        rows.append(\n            {\n                "method": method,\n                "final_train_loss": result["final_train_loss"],\n                "final_train_r2": result["final_train_r2"],\n                "final_test_loss": result["final_test_loss"],\n                "final_test_r2": result["final_test_r2"],\n                "best_test_loss": result["best_test_loss"],\n                "best_test_r2": result["best_test_r2"],\n                "diverged": result["diverged"],\n                "duration_sec": result["duration_sec"],\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef plot_training_histories(results_by_method):\n    fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)\n    for method in METHODS:\n        if method not in results_by_method:\n            continue\n        result = results_by_method[method]\n        history = result["history"]\n        axes[0].plot(history["epoch"], history["train_loss"], label=f"{METHOD_LABELS[method]} train", color=METHOD_COLORS[method])\n        axes[0].plot(history["epoch"], history["test_loss"], linestyle="--", label=f"{METHOD_LABELS[method]} test", color=METHOD_COLORS[method])\n        axes[1].plot(history["epoch"], history["train_r2"], label=f"{METHOD_LABELS[method]} train", color=METHOD_COLORS[method])\n        axes[1].plot(history["epoch"], history["test_r2"], linestyle="--", label=f"{METHOD_LABELS[method]} test", color=METHOD_COLORS[method])\n        axes[2].plot(history["epoch"], history["weight_norm"], label=METHOD_LABELS[method], color=METHOD_COLORS[method])\n    axes[0].set_title("Loss")\n    axes[0].set_ylabel("MSE")\n    axes[0].legend(ncol=2)\n    axes[1].set_title("R^2")\n    axes[1].set_ylabel("R^2")\n    axes[1].legend(ncol=2)\n    axes[2].set_title("Weight Norm")\n    axes[2].set_ylabel("L2 norm")\n    axes[2].set_xlabel("Epoch")\n    axes[2].legend(ncol=2)\n    fig.tight_layout()\n    plt.show()\n\ndef plot_layer_output_norm_histories(results_by_method):\n    first_result = next((results_by_method[method] for method in METHODS if method in results_by_method), None)\n    if first_result is None:\n        return\n\n    layer_output_history = first_result["history"].get("layer_output_norms", [])\n    if len(layer_output_history) == 0:\n        return\n\n    num_layers = len(layer_output_history[0])\n    fig, axes = plt.subplots(num_layers, 1, figsize=(10, 3.4 * num_layers), sharex=True, squeeze=False)\n    axes = axes.flatten()\n\n    for layer_idx in range(num_layers):\n        fan_in = int(first_result["state_dict"][f"layers.{layer_idx}.weight"].shape[1])\n        layer_label = "Input layer" if layer_idx == 0 else f"Hidden layer {layer_idx} output"\n        axis = axes[layer_idx]\n        for method in METHODS:\n            if method not in results_by_method:\n                continue\n            history = results_by_method[method]["history"]\n            values = [epoch_values[layer_idx] for epoch_values in history.get("layer_output_norms", [])]\n            if len(values) == 0:\n                continue\n            axis.plot(history["epoch"], values, label=METHOD_LABELS[method], color=METHOD_COLORS[method])\n        axis.set_title(f"{layer_label} norm / sqrt({fan_in} + 1)")\n        axis.set_ylabel("Average norm")\n        axis.legend(ncol=2)\n\n    axes[-1].set_xlabel("Epoch")\n    fig.tight_layout()\n    plt.show()\n\n\ndef load_california_housing(\n    test_size=0.2,\n    batch_size=256,\n    eval_batch_size=4096,\n    seed=0,\n    num_workers=0,\n    data_home=None,\n):\n    dataset = fetch_california_housing(data_home=data_home)\n    x_train, x_test, y_train, y_test = train_test_split(\n        dataset.data,\n        dataset.target,\n        test_size=test_size,\n        random_state=seed,\n    )\n\n    x_scaler = StandardScaler()\n    y_scaler = StandardScaler()\n\n    x_train = x_scaler.fit_transform(x_train)\n    x_test = x_scaler.transform(x_test)\n    y_train = y_scaler.fit_transform(y_train.reshape(-1, 1))\n    y_test = y_scaler.transform(y_test.reshape(-1, 1))\n\n    x_train = torch.tensor(x_train, dtype=torch.float32)\n    y_train = torch.tensor(y_train, dtype=torch.float32)\n    x_test = torch.tensor(x_test, dtype=torch.float32)\n    y_test = torch.tensor(y_test, dtype=torch.float32)\n\n    train_dataset = TensorDataset(x_train, y_train)\n    test_dataset = TensorDataset(x_test, y_test)\n\n    pin_memory = device.type == "cuda"\n    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)\n    train_eval_loader = DataLoader(train_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)\n    test_loader = DataLoader(test_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)\n\n    return {\n        "train_loader": train_loader,\n        "train_eval_loader": train_eval_loader,\n        "test_loader": test_loader,\n        "train_size": len(train_dataset),\n        "train_eval_size": len(train_dataset),\n        "test_size": len(test_dataset),\n        "x_train": x_train,\n        "y_train": y_train,\n        "x_test": x_test,\n        "y_test": y_test,\n    }\n\n\ndef plot_prediction_diagnostics(results_by_method, data, dimensions, device):\n    y_true = data["y_test"].squeeze().numpy()\n    min_val = float(np.min(y_true))\n    max_val = float(np.max(y_true))\n\n    plt.figure(figsize=(8, 8))\n    plt.plot([min_val, max_val], [min_val, max_val], color="black", linestyle="--", label="Perfect prediction")\n\n    for method in METHODS:\n        if method not in results_by_method:\n            continue\n        model = make_model(dimensions, require_grad=(method == "bp"), device=device)\n        model.load_state_dict(results_by_method[method]["state_dict"])\n        prediction = predict_tensor(model, data["x_test"], device).squeeze().numpy()\n        plt.scatter(y_true, prediction, s=10, alpha=0.25, color=METHOD_COLORS[method], label=METHOD_LABELS[method])\n\n    plt.title("California Housing Test Predictions")\n    plt.xlabel("True standardized target")\n    plt.ylabel("Predicted standardized target")\n    plt.legend()\n    plt.tight_layout()\n    plt.show()\n',
    'mnist': 'import itertools\nimport math\nimport random\nimport time\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom IPython.display import display\nfrom torch.utils.data import DataLoader, TensorDataset, Subset\nfrom torchvision.datasets import MNIST\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nif hasattr(torch, "set_float32_matmul_precision"):\n    torch.set_float32_matmul_precision("high")\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\nset_seed(0)\nprint(f"device={device}")\nif device.type == "cuda":\n    print(torch.cuda.get_device_name(0))\n\n\nMETHODS = ["bp", "np", "np_fan_in", "np_fixed", "wp"]\nMETHOD_LABELS = {\n    "bp": "BP",\n    "np": "IS-NP",\n    "np_fan_in": "Fan-in NP",\n    "np_fixed": "Vanilla NP",\n    "wp": "WP",\n}\nMETHOD_COLORS = {\n    "bp": "C0",\n    "np": "C1",\n    "np_fan_in": "C4",\n    "np_fixed": "C3",\n    "wp": "C2",\n}\n\n\nclass MLP(nn.Module):\n    def __init__(self, dimensions, activation=torch.relu, output_activation=None, require_grad=False):\n        super().__init__()\n        self.layers = nn.ModuleList(\n            [nn.Linear(dimensions[i], dimensions[i + 1], bias=True) for i in range(len(dimensions) - 1)]\n        )\n        self.activation = activation\n        self.output_activation = output_activation\n        if not require_grad:\n            for p in self.parameters():\n                p.requires_grad_(False)\n\n    def forward(self, x):\n        final_layer = len(self.layers) - 1\n        h = x\n        for i, layer in enumerate(self.layers):\n            u = layer(h)\n            if i == final_layer:\n                h = self.output_activation(u) if self.output_activation else u\n            else:\n                h = self.activation(u)\n        return h\n\n    def forward_weight_perturb(self, x, sigma):\n        batch_size = x.shape[0]\n        xs = [x]\n        ys = []\n        noises = []\n        h = x\n        for i, layer in enumerate(self.layers):\n            w = layer.weight\n            b = layer.bias\n            eps_w = torch.randn(batch_size, *w.shape, device=w.device, dtype=w.dtype) * sigma\n            eps_b = torch.randn(batch_size, *b.shape, device=b.device, dtype=b.dtype) * sigma\n            w_used = w.unsqueeze(0) + eps_w\n            b_used = b.unsqueeze(0) + eps_b\n            u = torch.bmm(h.unsqueeze(1), w_used.transpose(1, 2)).squeeze(1) + b_used\n            final_layer = len(self.layers) - 1\n            if i == final_layer:\n                h = self.output_activation(u) if self.output_activation else u\n                ys.append(h)\n            else:\n                h = self.activation(u)\n                ys.append(h)\n                xs.append(h)\n            noises.append((eps_w, eps_b))\n        return ys, xs, noises\n\n    def forward_node_perturb(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            x_in = a_noisy\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            noise_scale = sigma * torch.sqrt(1.0 + x_in.pow(2).sum(dim=1, keepdim=True))\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n    def forward_node_perturb_fixed_sigma(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            noise_scale = torch.full_like(z_noisy, sigma)\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n    def forward_node_perturb_fan_in_scaled(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            fan_in_with_bias = layer.in_features + 1\n            noise_scale_value = sigma * math.sqrt(fan_in_with_bias)\n            noise_scale = torch.full_like(z_noisy, noise_scale_value)\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n\ndef _mean_loss_per_sample(prediction, target):\n    loss = F.mse_loss(prediction, target, reduction="none")\n    if loss.dim() > 1:\n        loss = loss.mean(dim=1)\n    return loss.view(-1)\n\n\ndef _centered_reward_signal(loss_per_sample):\n    reward = -loss_per_sample\n    return reward - reward.mean()\n\n\ndef _flatten_parameter_tensors(weight_tensors, bias_tensors):\n    pieces = []\n    for weight_tensor, bias_tensor in zip(weight_tensors, bias_tensors):\n        pieces.append(weight_tensor.reshape(-1))\n        pieces.append(bias_tensor.reshape(-1))\n    return torch.cat(pieces)\n\n\ndef node_perturbation_step(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef node_perturbation_step_fixed_sigma(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fixed_sigma(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef node_perturbation_step_fan_in_scaled(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fan_in_scaled(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef weight_perturb_step(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    layer_outputs, _, noises = model.forward_weight_perturb(X, sigma)\n    prediction_noisy = layer_outputs[-1]\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    noise_scale = sigma ** 2 + 1e-12\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, (weight_noise, bias_noise) in zip(model.layers, noises):\n            scaled_weight_noise = scalar_signal.view(-1, 1, 1) * weight_noise / noise_scale\n            scaled_bias_noise = scalar_signal.view(-1, 1) * bias_noise / noise_scale\n            raw_weight_update = scaled_weight_noise.mean(dim=0)\n            raw_bias_update = scaled_bias_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef backprop_step(model, X, target, optimizer, loss_fn=F.mse_loss, return_unscaled_parameter_update_vector=False):\n    model.train()\n    for p in model.parameters():\n        p.requires_grad_(True)\n    optimizer.zero_grad()\n    y_pred = model(X)\n    loss = loss_fn(y_pred, target, reduction="mean")\n    loss.backward()\n    weight_grads = [layer.weight.grad.detach().clone() for layer in model.layers]\n    bias_grads = [layer.bias.grad.detach().clone() for layer in model.layers]\n    optimizer.step()\n    if return_unscaled_parameter_update_vector:\n        return loss.item(), -_flatten_parameter_tensors(weight_grads, bias_grads)\n    return loss.item()\n\n\ndef cosine_similarity_safe(a, b, eps=1e-12):\n    a_norm = torch.norm(a)\n    b_norm = torch.norm(b)\n    if a_norm.item() < eps or b_norm.item() < eps:\n        return 0.0\n    return float(torch.dot(a, b) / (a_norm * b_norm + eps))\n\n\ndef true_gradient_vector(model, xb, yb):\n    requires_grad_state = [parameter.requires_grad for parameter in model.parameters()]\n    for parameter in model.parameters():\n        parameter.requires_grad_(True)\n    model.zero_grad(set_to_none=True)\n    prediction = model(xb)\n    loss = F.mse_loss(prediction, yb, reduction="mean")\n    loss.backward()\n    weight_grads = [layer.weight.grad.detach().clone() for layer in model.layers]\n    bias_grads = [layer.bias.grad.detach().clone() for layer in model.layers]\n    flat_grad = _flatten_parameter_tensors(weight_grads, bias_grads)\n    model.zero_grad(set_to_none=True)\n    for parameter, old_value in zip(model.parameters(), requires_grad_state):\n        parameter.requires_grad_(old_value)\n    return -flat_grad\n\n\ndef gradient_metrics(unscaled_parameter_update_vector, true_update):\n    diff = unscaled_parameter_update_vector - true_update\n    cosine = cosine_similarity_safe(unscaled_parameter_update_vector, true_update)\n    squared_error = float(diff.pow(2).mean())\n    true_update_norm = float(torch.norm(true_update))\n    projection = float(torch.dot(unscaled_parameter_update_vector, true_update) / (true_update_norm + 1e-12))\n    return cosine, squared_error, projection\n\n\ndef node_perturbation_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef node_perturbation_fixed_sigma_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fixed_sigma(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef node_perturbation_fan_in_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fan_in_scaled(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef weight_perturbation_gradient_estimate_vector(model, xb, yb, sigma):\n    layer_outputs, _, noises = model.forward_weight_perturb(xb, sigma)\n    prediction_noisy = layer_outputs[-1]\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    noise_scale = sigma ** 2 + 1e-12\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for weight_noise, bias_noise in noises:\n        scaled_weight_noise = scalar_signal.view(-1, 1, 1) * weight_noise / noise_scale\n        scaled_bias_noise = scalar_signal.view(-1, 1) * bias_noise / noise_scale\n        raw_weight_updates.append(scaled_weight_noise.mean(dim=0))\n        raw_bias_updates.append(scaled_bias_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef perturbation_gradient_estimate_vector(method, model, xb, yb, sigma):\n    if method == "np":\n        return node_perturbation_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "np_fan_in":\n        return node_perturbation_fan_in_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "np_fixed":\n        return node_perturbation_fixed_sigma_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "wp":\n        return weight_perturbation_gradient_estimate_vector(model, xb, yb, sigma)\n    raise ValueError(f"Unknown perturbation method: {method}")\n\n\ndef variance_against_true_gradient(estimate_matrix, true_update):\n    num_samples = estimate_matrix.shape[0]\n    if num_samples == 0:\n        return 0.0, 0.0\n\n    squared_errors = (estimate_matrix - true_update.unsqueeze(0)).pow(2)\n    per_sample_variance = float(squared_errors.mean(dim=1).mean())\n    mean_estimate = estimate_matrix.mean(dim=0)\n    batch_variance = float((mean_estimate - true_update).pow(2).mean())\n    return per_sample_variance, batch_variance\n\n\ndef model_weight_norm(model):\n    total = torch.tensor(0.0, device=next(model.parameters()).device)\n    for layer in model.layers:\n        total = total + layer.weight.detach().pow(2).sum()\n    return float(torch.sqrt(total))\n\n\ndef mean_normalized_layer_output_norms(model, loader, device):\n    model.eval()\n    summed_norms = [0.0 for _ in model.layers]\n    total_examples = 0\n\n    with torch.no_grad():\n        for batch in loader:\n            if len(batch) == 3:\n                xb, _, _ = batch\n            else:\n                xb, _ = batch\n            xb = xb.to(device, non_blocking=True)\n            h = xb\n            batch_size = xb.size(0)\n\n            for layer_idx, layer in enumerate(model.layers):\n                h_for_norm = h if h.dim() > 1 else h.unsqueeze(1)\n                normalized_norm = torch.norm(h_for_norm, dim=1) / math.sqrt(h_for_norm.shape[1] + 1)\n                summed_norms[layer_idx] += normalized_norm.sum().item()\n                u = layer(h)\n                if layer_idx == len(model.layers) - 1:\n                    h = model.output_activation(u) if model.output_activation else u\n                else:\n                    h = model.activation(u)\n\n            total_examples += batch_size\n\n    return [value / max(total_examples, 1) for value in summed_norms]\n\n\ndef _random_subset_indices(n, limit, seed):\n    if limit is None or limit >= n:\n        return torch.arange(n)\n    g = torch.Generator().manual_seed(seed)\n    return torch.randperm(n, generator=g)[:limit]\n\n\ndef load_mnist(\n    train_limit=20000,\n    test_limit=5000,\n    train_eval_limit=4000,\n    batch_size=128,\n    eval_batch_size=1024,\n    data_dir="./data",\n    seed=0,\n    mean_center_only=True,\n    num_workers=0,\n):\n    train_dataset = MNIST(root=data_dir, train=True, download=True)\n    test_dataset = MNIST(root=data_dir, train=False, download=True)\n\n    x_train = train_dataset.data.float() / 255.0\n    x_test = test_dataset.data.float() / 255.0\n    train_labels = train_dataset.targets.long()\n    test_labels = test_dataset.targets.long()\n\n    train_idx = _random_subset_indices(len(x_train), train_limit, seed)\n    test_idx = _random_subset_indices(len(x_test), test_limit, seed + 1)\n\n    x_train = x_train[train_idx]\n    train_labels = train_labels[train_idx]\n    x_test = x_test[test_idx]\n    test_labels = test_labels[test_idx]\n\n    mean = x_train.mean()\n    if mean_center_only:\n        x_train = x_train - mean\n        x_test = x_test - mean\n    else:\n        std = x_train.std().clamp_min(1e-6)\n        x_train = (x_train - mean) / std\n        x_test = (x_test - mean) / std\n\n    x_train = x_train.view(x_train.size(0), -1)\n    x_test = x_test.view(x_test.size(0), -1)\n\n    y_train = F.one_hot(train_labels, num_classes=10).float()\n    y_test = F.one_hot(test_labels, num_classes=10).float()\n\n    train_tensor_dataset = TensorDataset(x_train, y_train, train_labels)\n    test_tensor_dataset = TensorDataset(x_test, y_test, test_labels)\n\n    if train_eval_limit is None or train_eval_limit >= len(train_tensor_dataset):\n        train_eval_dataset = train_tensor_dataset\n    else:\n        eval_idx = _random_subset_indices(len(train_tensor_dataset), train_eval_limit, seed + 2)\n        train_eval_dataset = Subset(train_tensor_dataset, eval_idx.tolist())\n\n    pin_memory = device.type == "cuda"\n    train_loader = DataLoader(train_tensor_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)\n    train_eval_loader = DataLoader(train_eval_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)\n    test_loader = DataLoader(test_tensor_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)\n\n    return {\n        "train_loader": train_loader,\n        "train_eval_loader": train_eval_loader,\n        "test_loader": test_loader,\n        "train_size": len(train_tensor_dataset),\n        "train_eval_size": len(train_eval_dataset),\n        "test_size": len(test_tensor_dataset),\n        "x_train": x_train,\n        "y_train": y_train,\n        "train_labels": train_labels,\n        "x_test": x_test,\n        "y_test": y_test,\n        "test_labels": test_labels,\n    }\n\n\ndef evaluate_model(model, loader, device):\n    model.eval()\n    total_loss = 0.0\n    total_correct = 0\n    total_examples = 0\n    with torch.no_grad():\n        for xb, yb, labels in loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            labels = labels.to(device, non_blocking=True)\n            logits = model(xb)\n            loss = F.mse_loss(logits, yb, reduction="mean")\n            total_loss += loss.item() * xb.size(0)\n            total_correct += (logits.argmax(dim=1) == labels).sum().item()\n            total_examples += xb.size(0)\n    mean_loss = total_loss / max(total_examples, 1)\n    accuracy = total_correct / max(total_examples, 1)\n    return mean_loss, accuracy\n\n\ndef make_model(dimensions, require_grad, device):\n    return MLP(dimensions, activation=torch.relu, output_activation=None, require_grad=require_grad).to(device)\n\n\ndef train_one_run(\n    method,\n    run_config,\n    data,\n    dimensions,\n    epochs,\n    seed,\n    device,\n    print_every_epoch=1,\n    divergence_loss_threshold=5.0,\n):\n    set_seed(seed)\n\n    base_model = make_model(dimensions, require_grad=True, device=device)\n    base_state = {name: tensor.detach().clone() for name, tensor in base_model.state_dict().items()}\n\n    require_grad = method == "bp"\n    model = make_model(dimensions, require_grad=require_grad, device=device)\n    model.load_state_dict(base_state)\n\n    optimizer = None\n    if method == "bp":\n        optimizer = torch.optim.SGD(model.parameters(), lr=run_config["lr"])\n\n    history = {\n        "epoch": [],\n        "train_loss": [],\n        "train_acc": [],\n        "test_loss": [],\n        "test_acc": [],\n        "weight_norm": [],\n        "layer_output_norms": [],\n    }\n\n    diverged = False\n    best_test_acc = -float("inf")\n    best_test_loss = float("inf")\n    start_time = time.time()\n\n    train_loader = data["train_loader"]\n    train_eval_loader = data["train_eval_loader"]\n    test_loader = data["test_loader"]\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        for xb, yb, _ in train_loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n\n            if method == "bp":\n                backprop_step(model, xb, yb, optimizer=optimizer)\n            elif method == "np":\n                node_perturbation_step(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "np_fan_in":\n                node_perturbation_step_fan_in_scaled(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "np_fixed":\n                node_perturbation_step_fixed_sigma(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "wp":\n                weight_perturb_step(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            else:\n                raise ValueError(f"Unknown method: {method}")\n\n        train_loss, train_acc = evaluate_model(model, train_eval_loader, device)\n        test_loss, test_acc = evaluate_model(model, test_loader, device)\n\n        history["epoch"].append(epoch)\n        history["train_loss"].append(train_loss)\n        history["train_acc"].append(train_acc)\n        history["test_loss"].append(test_loss)\n        history["test_acc"].append(test_acc)\n        history["weight_norm"].append(model_weight_norm(model))\n        history["layer_output_norms"].append(mean_normalized_layer_output_norms(model, train_eval_loader, device))\n\n        best_test_acc = max(best_test_acc, test_acc)\n        best_test_loss = min(best_test_loss, test_loss)\n\n        if (not math.isfinite(train_loss)) or (not math.isfinite(test_loss)) or test_loss > divergence_loss_threshold:\n            diverged = True\n            print(\n                f"    diverged at epoch {epoch}/{epochs} | "\n                f"train_loss={train_loss:.4f}, test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"\n            )\n            break\n\n        if epoch % print_every_epoch == 0 or epoch == 1 or epoch == epochs:\n            sigma_str = f", sigma={run_config[\'sigma\']:.4g}" if "sigma" in run_config else ""\n            print(\n                f"    epoch {epoch:2d}/{epochs} | {method} | lr={run_config[\'lr\']:.4g}{sigma_str} | "\n                f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "\n                f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"\n            )\n\n    duration_sec = time.time() - start_time\n    state_dict_cpu = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n    result = {\n        "method": method,\n        "lr": float(run_config["lr"]),\n        "sigma": float(run_config["sigma"]) if "sigma" in run_config else np.nan,\n        "epochs_completed": len(history["epoch"]),\n        "final_train_loss": history["train_loss"][-1] if history["train_loss"] else np.nan,\n        "final_train_acc": history["train_acc"][-1] if history["train_acc"] else np.nan,\n        "final_test_loss": history["test_loss"][-1] if history["test_loss"] else np.nan,\n        "final_test_acc": history["test_acc"][-1] if history["test_acc"] else np.nan,\n        "best_test_loss": best_test_loss,\n        "best_test_acc": best_test_acc,\n        "diverged": diverged,\n        "duration_sec": duration_sec,\n        "history": history,\n        "state_dict": state_dict_cpu,\n    }\n    del model\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return result\n\n\ndef train_backprop_checkpoint_states(data, dimensions, epochs, seed, device, lr, checkpoint_epochs, print_every_epoch=100):\n    checkpoint_epochs = sorted(set(int(epoch) for epoch in checkpoint_epochs))\n    set_seed(seed)\n    model = make_model(dimensions, require_grad=True, device=device)\n    optimizer = torch.optim.SGD(model.parameters(), lr=lr)\n    checkpoint_states = {}\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        for xb, yb, _ in data["train_loader"]:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            backprop_step(model, xb, yb, optimizer=optimizer)\n\n        if epoch in checkpoint_epochs:\n            checkpoint_states[epoch] = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n\n        if epoch % print_every_epoch == 0 or epoch == 1 or epoch == epochs or epoch in checkpoint_epochs:\n            train_loss, train_acc = evaluate_model(model, data["train_eval_loader"], device)\n            test_loss, test_acc = evaluate_model(model, data["test_loader"], device)\n            print(\n                f"    bp checkpoint training epoch {epoch:4d}/{epochs} | "\n                f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "\n                f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"\n            )\n\n    return checkpoint_states\n\n\ndef plot_layer_output_norm_histories(results_by_method):\n    first_result = next((results_by_method[method] for method in METHODS if method in results_by_method), None)\n    if first_result is None:\n        return\n\n    layer_output_history = first_result["history"].get("layer_output_norms", [])\n    if len(layer_output_history) == 0:\n        return\n\n    num_layers = len(layer_output_history[0])\n    fig, axes = plt.subplots(num_layers, 1, figsize=(10, 3.4 * num_layers), sharex=True, squeeze=False)\n    axes = axes.flatten()\n\n    for layer_idx in range(num_layers):\n        fan_in = int(first_result["state_dict"][f"layers.{layer_idx}.weight"].shape[1])\n        layer_label = "Input layer" if layer_idx == 0 else f"Hidden layer {layer_idx} output"\n        axis = axes[layer_idx]\n        for method in METHODS:\n            if method not in results_by_method:\n                continue\n            history = results_by_method[method]["history"]\n            values = [epoch_values[layer_idx] for epoch_values in history.get("layer_output_norms", [])]\n            if len(values) == 0:\n                continue\n            axis.plot(history["epoch"], values, label=METHOD_LABELS[method], color=METHOD_COLORS[method])\n        axis.set_title(f"{layer_label} norm / sqrt({fan_in} + 1)")\n        axis.set_ylabel("Average norm")\n        axis.legend(ncol=2)\n\n    axes[-1].set_xlabel("Epoch")\n    fig.tight_layout()\n    plt.show()\n\n\ndef layer_parameter_slices(model):\n    layer_slices = []\n    start = 0\n    for layer_idx, layer in enumerate(model.layers):\n        layer_parameter_count = layer.weight.numel() + layer.bias.numel()\n        layer_slices.append((layer_idx, slice(start, start + layer_parameter_count), f"Layer {layer_idx + 1}"))\n        start += layer_parameter_count\n    return layer_slices\n\n\ndef analyze_frozen_backprop_estimators(\n    checkpoint_states,\n    data,\n    dimensions,\n    device,\n    method_sigmas,\n    num_perturbations=50,\n    batch_size=128,\n):\n    analysis_dataset = TensorDataset(data["x_train"], data["y_train"])\n    analysis_loader = DataLoader(\n        analysis_dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=(device.type == "cuda"),\n    )\n\n    rows = []\n    perturbation_methods = list(method_sigmas.keys())\n\n    for checkpoint_epoch, state_dict in checkpoint_states.items():\n        model = make_model(dimensions, require_grad=True, device=device)\n        model.load_state_dict(state_dict)\n        component_specs = [(-1, slice(None), "All layers")] + layer_parameter_slices(model)\n\n        batch_metrics = {\n            method: {\n                layer_index: {\n                    "component": "all" if layer_index == -1 else f"layer_{layer_index + 1}",\n                    "component_label": component_label,\n                    "avg_sample_cosine": [],\n                    "mean_estimate_cosine": [],\n                    "sample_variance": [],\n                    "batch_variance": [],\n                }\n                for layer_index, _, component_label in component_specs\n            }\n            for method in perturbation_methods\n        }\n\n        for xb, yb in analysis_loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            true_update = true_gradient_vector(model, xb, yb)\n\n            for method in perturbation_methods:\n                sigma = method_sigmas[method]\n                estimates = [\n                    perturbation_gradient_estimate_vector(method, model, xb, yb, sigma)\n                    for _ in range(num_perturbations)\n                ]\n                estimate_matrix = torch.stack(estimates, dim=0)\n                mean_estimate = estimate_matrix.mean(dim=0)\n\n                for layer_index, component_slice, _ in component_specs:\n                    component_true_update = true_update[component_slice]\n                    component_estimate_matrix = estimate_matrix[:, component_slice]\n                    component_mean_estimate = mean_estimate[component_slice]\n\n                    avg_sample_cosine = float(\n                        np.mean(\n                            [\n                                cosine_similarity_safe(component_estimate, component_true_update)\n                                for component_estimate in component_estimate_matrix\n                            ]\n                        )\n                    )\n                    mean_estimate_cosine = cosine_similarity_safe(component_mean_estimate, component_true_update)\n                    sample_variance, batch_variance = variance_against_true_gradient(\n                        component_estimate_matrix,\n                        component_true_update,\n                    )\n\n                    metrics = batch_metrics[method][layer_index]\n                    metrics["avg_sample_cosine"].append(avg_sample_cosine)\n                    metrics["mean_estimate_cosine"].append(mean_estimate_cosine)\n                    metrics["sample_variance"].append(sample_variance)\n                    metrics["batch_variance"].append(batch_variance)\n\n        checkpoint_fraction = checkpoint_epoch / max(checkpoint_states.keys())\n        for method in perturbation_methods:\n            for layer_index, _, component_label in component_specs:\n                metrics = batch_metrics[method][layer_index]\n                rows.append({\n                    "checkpoint_epoch": checkpoint_epoch,\n                    "checkpoint_fraction": checkpoint_fraction,\n                    "method": method,\n                    "sigma": method_sigmas[method],\n                    "component": metrics["component"],\n                    "component_label": component_label,\n                    "layer_index": layer_index,\n                    "avg_sample_cosine": float(np.mean(metrics["avg_sample_cosine"])),\n                    "mean_estimate_cosine": float(np.mean(metrics["mean_estimate_cosine"])),\n                    "sample_variance": float(np.mean(metrics["sample_variance"])),\n                    "batch_variance": float(np.mean(metrics["batch_variance"])),\n                })\n\n    return pd.DataFrame(rows)\n\n\ndef plot_frozen_estimator_statistics(stats_df):\n    metrics = [\n        ("avg_sample_cosine", "Average Sample Cosine", "Cosine"),\n        ("mean_estimate_cosine", "Cosine of Mean Estimator", "Cosine"),\n        ("sample_variance", "Per-Sample Variance to True Gradient", "Mean squared distance per parameter"),\n        ("batch_variance", "Batch-Average Variance to True Gradient", "Mean squared distance per parameter"),\n    ]\n    perturbation_methods = [method for method in METHODS if method in {"np", "np_fan_in", "np_fixed", "wp"}]\n\n    overall_df = stats_df[stats_df["layer_index"] == -1] if "layer_index" in stats_df.columns else stats_df\n    fig, axes = plt.subplots(1, len(metrics), figsize=(24, 4), sharex=True)\n\n    for method in perturbation_methods:\n        method_df = overall_df[overall_df["method"] == method].sort_values("checkpoint_epoch")\n        if len(method_df) == 0:\n            continue\n        x = method_df["checkpoint_epoch"].to_numpy()\n        for axis, (column, title, ylabel) in zip(axes, metrics):\n            axis.plot(x, method_df[column].to_numpy(), marker="o", color=METHOD_COLORS[method], label=METHOD_LABELS[method])\n            axis.set_title(f"All layers | {title}")\n            axis.set_xlabel("Checkpoint epoch")\n            axis.set_ylabel(ylabel)\n            axis.legend()\n\n    fig.tight_layout()\n    plt.show()\n\n    if "layer_index" not in stats_df.columns:\n        return\n\n    layer_indices = sorted(layer_index for layer_index in stats_df["layer_index"].unique() if layer_index >= 0)\n    if len(layer_indices) == 0:\n        return\n\n    fig, axes = plt.subplots(\n        len(layer_indices),\n        len(metrics),\n        figsize=(6 * len(metrics), 3.8 * len(layer_indices)),\n        sharex=True,\n        squeeze=False,\n    )\n\n    for row, layer_index in enumerate(layer_indices):\n        layer_df = stats_df[stats_df["layer_index"] == layer_index]\n        component_label = layer_df["component_label"].iloc[0]\n\n        for col, (column, title, ylabel) in enumerate(metrics):\n            axis = axes[row, col]\n            for method in perturbation_methods:\n                method_df = layer_df[layer_df["method"] == method].sort_values("checkpoint_epoch")\n                if len(method_df) == 0:\n                    continue\n                axis.plot(\n                    method_df["checkpoint_epoch"].to_numpy(),\n                    method_df[column].to_numpy(),\n                    marker="o",\n                    color=METHOD_COLORS[method],\n                    label=METHOD_LABELS[method],\n                )\n            axis.set_title(f"{component_label} | {title}")\n            axis.set_xlabel("Checkpoint epoch")\n            axis.set_ylabel(ylabel)\n            if col == len(metrics) - 1:\n                axis.legend(fontsize=8)\n\n    fig.tight_layout()\n    plt.show()\n\n\ndef analyze_frozen_backprop_sigma_grid(\n    checkpoint_states,\n    data,\n    dimensions,\n    device,\n    method_sigma_grid,\n    num_perturbations=50,\n    batch_size=128,\n):\n    analysis_dataset = TensorDataset(data["x_train"], data["y_train"])\n    analysis_loader = DataLoader(\n        analysis_dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=(device.type == "cuda"),\n    )\n\n    rows = []\n    methods = list(method_sigma_grid.keys())\n\n    for checkpoint_epoch, state_dict in checkpoint_states.items():\n        model = make_model(dimensions, require_grad=True, device=device)\n        model.load_state_dict(state_dict)\n\n        batch_metrics = {\n            (method, sigma): {\n                "avg_sample_cosine": [],\n                "mean_estimate_cosine": [],\n                "sample_variance": [],\n                "batch_variance": [],\n            }\n            for method in methods\n            for sigma in method_sigma_grid[method]\n        }\n\n        for batch in analysis_loader:\n            if len(batch) == 3:\n                xb, yb, _ = batch\n            else:\n                xb, yb = batch\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            true_update = true_gradient_vector(model, xb, yb)\n\n            for method in methods:\n                for sigma in method_sigma_grid[method]:\n                    estimates = [\n                        perturbation_gradient_estimate_vector(method, model, xb, yb, sigma)\n                        for _ in range(num_perturbations)\n                    ]\n                    estimate_matrix = torch.stack(estimates, dim=0)\n                    mean_estimate = estimate_matrix.mean(dim=0)\n\n                    avg_sample_cosine = float(\n                        sum(cosine_similarity_safe(estimate, true_update) for estimate in estimates) / num_perturbations\n                    )\n                    mean_estimate_cosine = cosine_similarity_safe(mean_estimate, true_update)\n                    sample_variance, batch_variance = variance_against_true_gradient(estimate_matrix, true_update)\n\n                    metrics = batch_metrics[(method, sigma)]\n                    metrics["avg_sample_cosine"].append(avg_sample_cosine)\n                    metrics["mean_estimate_cosine"].append(mean_estimate_cosine)\n                    metrics["sample_variance"].append(sample_variance)\n                    metrics["batch_variance"].append(batch_variance)\n\n        checkpoint_fraction = checkpoint_epoch / max(checkpoint_states.keys())\n        for method in methods:\n            for sigma in method_sigma_grid[method]:\n                metrics = batch_metrics[(method, sigma)]\n                rows.append({\n                    "checkpoint_epoch": checkpoint_epoch,\n                    "checkpoint_fraction": checkpoint_fraction,\n                    "method": method,\n                    "sigma": float(sigma),\n                    "avg_sample_cosine": float(np.mean(metrics["avg_sample_cosine"])),\n                    "mean_estimate_cosine": float(np.mean(metrics["mean_estimate_cosine"])),\n                    "sample_variance": float(np.mean(metrics["sample_variance"])),\n                    "batch_variance": float(np.mean(metrics["batch_variance"])),\n                })\n\n    return pd.DataFrame(rows)\n\n\ndef plot_frozen_sigma_search_results(stats_df):\n    metrics = [\n        ("avg_sample_cosine", "Average Sample Cosine", "Cosine"),\n        ("mean_estimate_cosine", "Cosine of Mean Estimator", "Cosine"),\n        ("sample_variance", "Per-Sample Variance to True Gradient", "Mean squared distance per parameter"),\n        ("batch_variance", "Batch-Average Variance to True Gradient", "Mean squared distance per parameter"),\n    ]\n    checkpoint_epochs = sorted(stats_df["checkpoint_epoch"].unique())\n\n    for method in [method for method in METHODS if method in set(stats_df["method"])] :\n        method_df = stats_df[stats_df["method"] == method]\n        fig, axes = plt.subplots(\n            len(checkpoint_epochs),\n            len(metrics),\n            figsize=(5 * len(metrics), 3.8 * len(checkpoint_epochs)),\n            squeeze=False,\n        )\n\n        for row, checkpoint_epoch in enumerate(checkpoint_epochs):\n            subset = method_df[method_df["checkpoint_epoch"] == checkpoint_epoch].sort_values("sigma")\n            sigma_labels = [f"{sigma:.4g}" for sigma in subset["sigma"].to_numpy()]\n\n            for col, (column, title, ylabel) in enumerate(metrics):\n                axis = axes[row, col]\n                axis.bar(sigma_labels, subset[column].to_numpy(), color=METHOD_COLORS[method])\n                axis.set_title(f"epoch {checkpoint_epoch} | {title}")\n                axis.set_xlabel("sigma")\n                axis.set_ylabel(ylabel)\n                axis.tick_params(axis="x", rotation=45)\n\n        fig.suptitle(f"{METHOD_LABELS[method]} sigma search", y=1.02)\n        fig.tight_layout()\n        plt.show()\n\n\ndef build_grid(space):\n    keys = list(space.keys())\n    values = [space[key] for key in keys]\n    return [dict(zip(keys, combo)) for combo in itertools.product(*values)]\n\n\ndef run_sweep(search_spaces, data, dimensions, sweep_epochs, seeds, device):\n    results = []\n    flat_records = []\n    total_runs = sum(len(configs) * len(seeds) for configs in search_spaces.values())\n    run_idx = 0\n\n    print(f"Starting sweep with total_runs={total_runs}, device={device}")\n    for method in METHODS:\n        configs = search_spaces[method]\n        for config in configs:\n            for seed in seeds:\n                run_idx += 1\n                sigma_str = f", sigma={config[\'sigma\']:.4g}" if "sigma" in config else ""\n                print(f"\\n=== Run {run_idx}/{total_runs} | {method} | lr={config[\'lr\']:.4g}{sigma_str} | seed={seed} ===")\n                result = train_one_run(\n                    method=method,\n                    run_config=config,\n                    data=data,\n                    dimensions=dimensions,\n                    epochs=sweep_epochs,\n                    seed=seed,\n                    device=device,\n                    print_every_epoch=1,\n                )\n                results.append(result)\n                flat_record = {k: v for k, v in result.items() if k not in {"history", "state_dict"}}\n                flat_record["seed"] = seed\n                flat_records.append(flat_record)\n\n    df = pd.DataFrame(flat_records)\n    if len(df) > 0:\n        df = df.sort_values(\n            ["method", "final_test_acc", "best_test_acc", "final_test_loss"],\n            ascending=[True, False, False, True],\n        ).reset_index(drop=True)\n    return results, df\n\n\ndef summarize_top_configs(df, top_k=5):\n    if len(df) == 0:\n        return df\n    frames = []\n    for method, group in df.groupby("method"):\n        frames.append(group.head(top_k))\n    return pd.concat(frames, ignore_index=True)\n\n\ndef select_best_configs(df):\n    rows = []\n    good = df[df["diverged"] == False]\n    for method, group in good.groupby("method"):\n        best = group.sort_values(\n            ["final_test_acc", "best_test_acc", "final_test_loss"],\n            ascending=[False, False, True],\n        ).iloc[0]\n        rows.append(best)\n    return pd.DataFrame(rows).reset_index(drop=True)\n\n\ndef run_best_config_comparison(best_df, data, dimensions, final_epochs, seed, device):\n    comparison_results = {}\n    for row in best_df.itertuples(index=False):\n        config = {"lr": float(row.lr)}\n        if not pd.isna(row.sigma):\n            config["sigma"] = float(row.sigma)\n        print(\n            f"\\n### Final run | {row.method} | lr={config[\'lr\']:.4g}" +\n            (f", sigma={config[\'sigma\']:.4g}" if \'sigma\' in config else "")\n        )\n        comparison_results[row.method] = train_one_run(\n            method=row.method,\n            run_config=config,\n            data=data,\n            dimensions=dimensions,\n            epochs=final_epochs,\n            seed=seed,\n            device=device,\n            print_every_epoch=1,\n        )\n    return comparison_results\n\n\ndef results_table(results_by_method):\n    rows = []\n    for method in METHODS:\n        if method not in results_by_method:\n            continue\n        result = results_by_method[method]\n        rows.append(\n            {\n                "method": method,\n                "final_train_loss": result["final_train_loss"],\n                "final_train_acc": result["final_train_acc"],\n                "final_test_loss": result["final_test_loss"],\n                "final_test_acc": result["final_test_acc"],\n                "best_test_loss": result["best_test_loss"],\n                "best_test_acc": result["best_test_acc"],\n                "diverged": result["diverged"],\n                "duration_sec": result["duration_sec"],\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef plot_training_histories(results_by_method):\n    fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)\n    for method in METHODS:\n        if method not in results_by_method:\n            continue\n        result = results_by_method[method]\n        history = result["history"]\n        axes[0].plot(history["epoch"], history["train_loss"], label=f"{METHOD_LABELS[method]} train", color=METHOD_COLORS[method])\n        axes[0].plot(history["epoch"], history["test_loss"], linestyle="--", label=f"{METHOD_LABELS[method]} test", color=METHOD_COLORS[method])\n        axes[1].plot(history["epoch"], history["train_acc"], label=f"{METHOD_LABELS[method]} train", color=METHOD_COLORS[method])\n        axes[1].plot(history["epoch"], history["test_acc"], linestyle="--", label=f"{METHOD_LABELS[method]} test", color=METHOD_COLORS[method])\n        axes[2].plot(history["epoch"], history["weight_norm"], label=METHOD_LABELS[method], color=METHOD_COLORS[method])\n    axes[0].set_title("Loss")\n    axes[0].set_ylabel("MSE")\n    axes[0].legend(ncol=2)\n    axes[1].set_title("Accuracy")\n    axes[1].set_ylabel("Accuracy")\n    axes[1].set_ylim(0.0, 1.0)\n    axes[1].legend(ncol=2)\n    axes[2].set_title("Weight Norm")\n    axes[2].set_ylabel("L2 norm")\n    axes[2].set_xlabel("Epoch")\n    axes[2].legend(ncol=2)\n    fig.tight_layout()\n    plt.show()\n',
    'cifar10': 'import itertools\nimport math\nimport random\nimport time\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom IPython.display import display\nfrom torch.utils.data import DataLoader, TensorDataset, Subset\nfrom torchvision.datasets import CIFAR10\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nif hasattr(torch, "set_float32_matmul_precision"):\n    torch.set_float32_matmul_precision("high")\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\nset_seed(0)\nprint(f"device={device}")\nif device.type == "cuda":\n    print(torch.cuda.get_device_name(0))\n\n\nMETHODS = ["bp", "np", "np_fan_in", "np_fixed", "wp"]\nMETHOD_LABELS = {\n    "bp": "BP",\n    "np": "IS-NP",\n    "np_fan_in": "Fan-in NP",\n    "np_fixed": "Vanilla NP",\n    "wp": "WP",\n}\nMETHOD_COLORS = {\n    "bp": "C0",\n    "np": "C1",\n    "np_fan_in": "C4",\n    "np_fixed": "C3",\n    "wp": "C2",\n}\n\n\nclass MLP(nn.Module):\n    def __init__(self, dimensions, activation=torch.relu, output_activation=None, require_grad=False):\n        super().__init__()\n        self.layers = nn.ModuleList(\n            [nn.Linear(dimensions[i], dimensions[i + 1], bias=True) for i in range(len(dimensions) - 1)]\n        )\n        self.activation = activation\n        self.output_activation = output_activation\n        if not require_grad:\n            for p in self.parameters():\n                p.requires_grad_(False)\n\n    def forward(self, x):\n        final_layer = len(self.layers) - 1\n        h = x\n        for i, layer in enumerate(self.layers):\n            u = layer(h)\n            if i == final_layer:\n                h = self.output_activation(u) if self.output_activation else u\n            else:\n                h = self.activation(u)\n        return h\n\n    def forward_weight_perturb(self, x, sigma):\n        batch_size = x.shape[0]\n        xs = [x]\n        ys = []\n        noises = []\n        h = x\n        for i, layer in enumerate(self.layers):\n            w = layer.weight\n            b = layer.bias\n            eps_w = torch.randn(batch_size, *w.shape, device=w.device, dtype=w.dtype) * sigma\n            eps_b = torch.randn(batch_size, *b.shape, device=b.device, dtype=b.dtype) * sigma\n            w_used = w.unsqueeze(0) + eps_w\n            b_used = b.unsqueeze(0) + eps_b\n            u = torch.bmm(h.unsqueeze(1), w_used.transpose(1, 2)).squeeze(1) + b_used\n            final_layer = len(self.layers) - 1\n            if i == final_layer:\n                h = self.output_activation(u) if self.output_activation else u\n                ys.append(h)\n            else:\n                h = self.activation(u)\n                ys.append(h)\n                xs.append(h)\n            noises.append((eps_w, eps_b))\n        return ys, xs, noises\n\n    def forward_node_perturb(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            x_in = a_noisy\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            noise_scale = sigma * torch.sqrt(1.0 + x_in.pow(2).sum(dim=1, keepdim=True))\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n    def forward_node_perturb_fixed_sigma(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            noise_scale = torch.full_like(z_noisy, sigma)\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n    def forward_node_perturb_fan_in_scaled(self, x, sigma):\n        acts = [x]\n        noises = []\n        noise_scales = []\n        last = len(self.layers) - 1\n        a_noisy = x\n        for i, layer in enumerate(self.layers):\n            z_noisy = layer(a_noisy)\n            eps = torch.randn_like(z_noisy)\n            fan_in_with_bias = layer.in_features + 1\n            noise_scale_value = sigma * math.sqrt(fan_in_with_bias)\n            noise_scale = torch.full_like(z_noisy, noise_scale_value)\n            noises.append(eps)\n            noise_scales.append(noise_scale)\n            z_noisy = z_noisy + noise_scale * eps\n            if i == last:\n                a_noisy = self.output_activation(z_noisy) if self.output_activation else z_noisy\n            else:\n                a_noisy = self.activation(z_noisy)\n            acts.append(a_noisy)\n        return acts, noises, noise_scales, a_noisy\n\n\ndef _mean_loss_per_sample(prediction, target):\n    loss = F.mse_loss(prediction, target, reduction="none")\n    if loss.dim() > 1:\n        loss = loss.mean(dim=1)\n    return loss.view(-1)\n\n\ndef _centered_reward_signal(loss_per_sample):\n    reward = -loss_per_sample\n    return reward - reward.mean()\n\n\ndef _flatten_parameter_tensors(weight_tensors, bias_tensors):\n    pieces = []\n    for weight_tensor, bias_tensor in zip(weight_tensors, bias_tensors):\n        pieces.append(weight_tensor.reshape(-1))\n        pieces.append(bias_tensor.reshape(-1))\n    return torch.cat(pieces)\n\n\ndef node_perturbation_step(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef node_perturbation_step_fixed_sigma(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fixed_sigma(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef node_perturbation_step_fan_in_scaled(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fan_in_scaled(X, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, x_in, noise, noise_scale in zip(model.layers, activations, noises, noise_scales):\n            scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n            raw_weight_update = torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0)\n            raw_bias_update = scaled_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef weight_perturb_step(model, X, target, eta=0.5, sigma=0.1, return_unscaled_parameter_update_vector=False):\n    model.train()\n    layer_outputs, _, noises = model.forward_weight_perturb(X, sigma)\n    prediction_noisy = layer_outputs[-1]\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, target)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    noise_scale = sigma ** 2 + 1e-12\n    raw_weight_updates = []\n    raw_bias_updates = []\n    with torch.no_grad():\n        for layer, (weight_noise, bias_noise) in zip(model.layers, noises):\n            scaled_weight_noise = scalar_signal.view(-1, 1, 1) * weight_noise / noise_scale\n            scaled_bias_noise = scalar_signal.view(-1, 1) * bias_noise / noise_scale\n            raw_weight_update = scaled_weight_noise.mean(dim=0)\n            raw_bias_update = scaled_bias_noise.mean(dim=0)\n            raw_weight_updates.append(raw_weight_update)\n            raw_bias_updates.append(raw_bias_update)\n            layer.weight += eta * raw_weight_update\n            layer.bias += eta * raw_bias_update\n    mean_loss = loss_per_sample.mean().item()\n    if return_unscaled_parameter_update_vector:\n        return mean_loss, _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n    return mean_loss\n\n\ndef backprop_step(model, X, target, optimizer, loss_fn=F.mse_loss, return_unscaled_parameter_update_vector=False):\n    model.train()\n    for p in model.parameters():\n        p.requires_grad_(True)\n    optimizer.zero_grad()\n    y_pred = model(X)\n    loss = loss_fn(y_pred, target, reduction="mean")\n    loss.backward()\n    weight_grads = [layer.weight.grad.detach().clone() for layer in model.layers]\n    bias_grads = [layer.bias.grad.detach().clone() for layer in model.layers]\n    optimizer.step()\n    if return_unscaled_parameter_update_vector:\n        return loss.item(), -_flatten_parameter_tensors(weight_grads, bias_grads)\n    return loss.item()\n\n\ndef cosine_similarity_safe(a, b, eps=1e-12):\n    a_norm = torch.norm(a)\n    b_norm = torch.norm(b)\n    if a_norm.item() < eps or b_norm.item() < eps:\n        return 0.0\n    return float(torch.dot(a, b) / (a_norm * b_norm + eps))\n\n\ndef true_gradient_vector(model, xb, yb):\n    requires_grad_state = [parameter.requires_grad for parameter in model.parameters()]\n    for parameter in model.parameters():\n        parameter.requires_grad_(True)\n    model.zero_grad(set_to_none=True)\n    prediction = model(xb)\n    loss = F.mse_loss(prediction, yb, reduction="mean")\n    loss.backward()\n    weight_grads = [layer.weight.grad.detach().clone() for layer in model.layers]\n    bias_grads = [layer.bias.grad.detach().clone() for layer in model.layers]\n    flat_grad = _flatten_parameter_tensors(weight_grads, bias_grads)\n    model.zero_grad(set_to_none=True)\n    for parameter, old_value in zip(model.parameters(), requires_grad_state):\n        parameter.requires_grad_(old_value)\n    return -flat_grad\n\n\ndef gradient_metrics(unscaled_parameter_update_vector, true_update):\n    diff = unscaled_parameter_update_vector - true_update\n    cosine = cosine_similarity_safe(unscaled_parameter_update_vector, true_update)\n    squared_error = float(diff.pow(2).mean())\n    true_update_norm = float(torch.norm(true_update))\n    projection = float(torch.dot(unscaled_parameter_update_vector, true_update) / (true_update_norm + 1e-12))\n    return cosine, squared_error, projection\n\n\ndef node_perturbation_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef node_perturbation_fixed_sigma_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fixed_sigma(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef node_perturbation_fan_in_gradient_estimate_vector(model, xb, yb, sigma):\n    activations, noises, noise_scales, prediction_noisy = model.forward_node_perturb_fan_in_scaled(xb, sigma)\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for x_in, noise, noise_scale in zip(activations, noises, noise_scales):\n        scaled_noise = scalar_signal.view(-1, 1) * noise / (noise_scale + 1e-12)\n        raw_weight_updates.append(torch.bmm(scaled_noise.unsqueeze(2), x_in.unsqueeze(1)).mean(dim=0))\n        raw_bias_updates.append(scaled_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef weight_perturbation_gradient_estimate_vector(model, xb, yb, sigma):\n    layer_outputs, _, noises = model.forward_weight_perturb(xb, sigma)\n    prediction_noisy = layer_outputs[-1]\n    loss_per_sample = _mean_loss_per_sample(prediction_noisy, yb)\n    scalar_signal = _centered_reward_signal(loss_per_sample)\n    noise_scale = sigma ** 2 + 1e-12\n    raw_weight_updates = []\n    raw_bias_updates = []\n    for weight_noise, bias_noise in noises:\n        scaled_weight_noise = scalar_signal.view(-1, 1, 1) * weight_noise / noise_scale\n        scaled_bias_noise = scalar_signal.view(-1, 1) * bias_noise / noise_scale\n        raw_weight_updates.append(scaled_weight_noise.mean(dim=0))\n        raw_bias_updates.append(scaled_bias_noise.mean(dim=0))\n    return _flatten_parameter_tensors(raw_weight_updates, raw_bias_updates)\n\n\ndef perturbation_gradient_estimate_vector(method, model, xb, yb, sigma):\n    if method == "np":\n        return node_perturbation_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "np_fan_in":\n        return node_perturbation_fan_in_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "np_fixed":\n        return node_perturbation_fixed_sigma_gradient_estimate_vector(model, xb, yb, sigma)\n    if method == "wp":\n        return weight_perturbation_gradient_estimate_vector(model, xb, yb, sigma)\n    raise ValueError(f"Unknown perturbation method: {method}")\n\n\ndef variance_against_true_gradient(estimate_matrix, true_update):\n    num_samples = estimate_matrix.shape[0]\n    if num_samples == 0:\n        return 0.0, 0.0\n\n    squared_errors = (estimate_matrix - true_update.unsqueeze(0)).pow(2)\n    per_sample_variance = float(squared_errors.mean(dim=1).mean())\n    mean_estimate = estimate_matrix.mean(dim=0)\n    batch_variance = float((mean_estimate - true_update).pow(2).mean())\n    return per_sample_variance, batch_variance\n\n\ndef model_weight_norm(model):\n    total = torch.tensor(0.0, device=next(model.parameters()).device)\n    for layer in model.layers:\n        total = total + layer.weight.detach().pow(2).sum()\n    return float(torch.sqrt(total))\n\n\ndef mean_normalized_layer_output_norms(model, loader, device):\n    model.eval()\n    summed_norms = [0.0 for _ in model.layers]\n    total_examples = 0\n\n    with torch.no_grad():\n        for batch in loader:\n            if len(batch) == 3:\n                xb, _, _ = batch\n            else:\n                xb, _ = batch\n            xb = xb.to(device, non_blocking=True)\n            h = xb\n            batch_size = xb.size(0)\n\n            for layer_idx, layer in enumerate(model.layers):\n                h_for_norm = h if h.dim() > 1 else h.unsqueeze(1)\n                normalized_norm = torch.norm(h_for_norm, dim=1) / math.sqrt(h_for_norm.shape[1] + 1)\n                summed_norms[layer_idx] += normalized_norm.sum().item()\n                u = layer(h)\n                if layer_idx == len(model.layers) - 1:\n                    h = model.output_activation(u) if model.output_activation else u\n                else:\n                    h = model.activation(u)\n\n            total_examples += batch_size\n\n    return [value / max(total_examples, 1) for value in summed_norms]\n\n\ndef _random_subset_indices(n, limit, seed):\n    if limit is None or limit >= n:\n        return torch.arange(n)\n    g = torch.Generator().manual_seed(seed)\n    return torch.randperm(n, generator=g)[:limit]\n\n\ndef load_cifar10(\n    train_limit=20000,\n    test_limit=5000,\n    train_eval_limit=4000,\n    batch_size=128,\n    eval_batch_size=1024,\n    data_dir="./data",\n    seed=0,\n    mean_center_only=True,\n    num_workers=0,\n):\n    train_dataset = CIFAR10(root=data_dir, train=True, download=True)\n    test_dataset = CIFAR10(root=data_dir, train=False, download=True)\n\n    x_train = torch.from_numpy(train_dataset.data).float() / 255.0\n    x_test = torch.from_numpy(test_dataset.data).float() / 255.0\n    train_labels = torch.tensor(train_dataset.targets, dtype=torch.long)\n    test_labels = torch.tensor(test_dataset.targets, dtype=torch.long)\n\n    train_idx = _random_subset_indices(len(x_train), train_limit, seed)\n    test_idx = _random_subset_indices(len(x_test), test_limit, seed + 1)\n\n    x_train = x_train[train_idx]\n    train_labels = train_labels[train_idx]\n    x_test = x_test[test_idx]\n    test_labels = test_labels[test_idx]\n\n    mean = x_train.mean()\n    if mean_center_only:\n        x_train = x_train - mean\n        x_test = x_test - mean\n    else:\n        std = x_train.std().clamp_min(1e-6)\n        x_train = (x_train - mean) / std\n        x_test = (x_test - mean) / std\n\n    x_train = x_train.permute(0, 3, 1, 2).contiguous().view(x_train.size(0), -1)\n    x_test = x_test.permute(0, 3, 1, 2).contiguous().view(x_test.size(0), -1)\n\n    y_train = F.one_hot(train_labels, num_classes=10).float()\n    y_test = F.one_hot(test_labels, num_classes=10).float()\n\n    train_tensor_dataset = TensorDataset(x_train, y_train, train_labels)\n    test_tensor_dataset = TensorDataset(x_test, y_test, test_labels)\n\n    if train_eval_limit is None or train_eval_limit >= len(train_tensor_dataset):\n        train_eval_dataset = train_tensor_dataset\n    else:\n        eval_idx = _random_subset_indices(len(train_tensor_dataset), train_eval_limit, seed + 2)\n        train_eval_dataset = Subset(train_tensor_dataset, eval_idx.tolist())\n\n    pin_memory = device.type == "cuda"\n    train_loader = DataLoader(train_tensor_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)\n    train_eval_loader = DataLoader(train_eval_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)\n    test_loader = DataLoader(test_tensor_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)\n\n    return {\n        "train_loader": train_loader,\n        "train_eval_loader": train_eval_loader,\n        "test_loader": test_loader,\n        "train_size": len(train_tensor_dataset),\n        "train_eval_size": len(train_eval_dataset),\n        "test_size": len(test_tensor_dataset),\n        "x_train": x_train,\n        "y_train": y_train,\n        "train_labels": train_labels,\n        "x_test": x_test,\n        "y_test": y_test,\n        "test_labels": test_labels,\n    }\n\n\ndef evaluate_model(model, loader, device):\n    model.eval()\n    total_loss = 0.0\n    total_correct = 0\n    total_examples = 0\n    with torch.no_grad():\n        for xb, yb, labels in loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            labels = labels.to(device, non_blocking=True)\n            logits = model(xb)\n            loss = F.mse_loss(logits, yb, reduction="mean")\n            total_loss += loss.item() * xb.size(0)\n            total_correct += (logits.argmax(dim=1) == labels).sum().item()\n            total_examples += xb.size(0)\n    mean_loss = total_loss / max(total_examples, 1)\n    accuracy = total_correct / max(total_examples, 1)\n    return mean_loss, accuracy\n\n\ndef make_model(dimensions, require_grad, device):\n    return MLP(dimensions, activation=torch.relu, output_activation=None, require_grad=require_grad).to(device)\n\n\ndef train_one_run(\n    method,\n    run_config,\n    data,\n    dimensions,\n    epochs,\n    seed,\n    device,\n    print_every_epoch=1,\n    divergence_loss_threshold=5.0,\n):\n    set_seed(seed)\n\n    base_model = make_model(dimensions, require_grad=True, device=device)\n    base_state = {name: tensor.detach().clone() for name, tensor in base_model.state_dict().items()}\n\n    require_grad = method == "bp"\n    model = make_model(dimensions, require_grad=require_grad, device=device)\n    model.load_state_dict(base_state)\n\n    optimizer = None\n    if method == "bp":\n        optimizer = torch.optim.SGD(model.parameters(), lr=run_config["lr"])\n\n    history = {\n        "epoch": [],\n        "train_loss": [],\n        "train_acc": [],\n        "test_loss": [],\n        "test_acc": [],\n        "weight_norm": [],\n        "layer_output_norms": [],\n    }\n\n    diverged = False\n    best_test_acc = -float("inf")\n    best_test_loss = float("inf")\n    start_time = time.time()\n\n    train_loader = data["train_loader"]\n    train_eval_loader = data["train_eval_loader"]\n    test_loader = data["test_loader"]\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        for xb, yb, _ in train_loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n\n            if method == "bp":\n                backprop_step(model, xb, yb, optimizer=optimizer)\n            elif method == "np":\n                node_perturbation_step(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "np_fan_in":\n                node_perturbation_step_fan_in_scaled(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "np_fixed":\n                node_perturbation_step_fixed_sigma(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            elif method == "wp":\n                weight_perturb_step(model, xb, yb, eta=run_config["lr"], sigma=run_config["sigma"])\n            else:\n                raise ValueError(f"Unknown method: {method}")\n\n        train_loss, train_acc = evaluate_model(model, train_eval_loader, device)\n        test_loss, test_acc = evaluate_model(model, test_loader, device)\n\n        history["epoch"].append(epoch)\n        history["train_loss"].append(train_loss)\n        history["train_acc"].append(train_acc)\n        history["test_loss"].append(test_loss)\n        history["test_acc"].append(test_acc)\n        history["weight_norm"].append(model_weight_norm(model))\n        history["layer_output_norms"].append(mean_normalized_layer_output_norms(model, train_eval_loader, device))\n\n        best_test_acc = max(best_test_acc, test_acc)\n        best_test_loss = min(best_test_loss, test_loss)\n\n        if (not math.isfinite(train_loss)) or (not math.isfinite(test_loss)) or test_loss > divergence_loss_threshold:\n            diverged = True\n            print(\n                f"    diverged at epoch {epoch}/{epochs} | "\n                f"train_loss={train_loss:.4f}, test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"\n            )\n            break\n\n        if epoch % print_every_epoch == 0 or epoch == 1 or epoch == epochs:\n            sigma_str = f", sigma={run_config[\'sigma\']:.4g}" if "sigma" in run_config else ""\n            print(\n                f"    epoch {epoch:2d}/{epochs} | {method} | lr={run_config[\'lr\']:.4g}{sigma_str} | "\n                f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "\n                f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"\n            )\n\n    duration_sec = time.time() - start_time\n    state_dict_cpu = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n    result = {\n        "method": method,\n        "lr": float(run_config["lr"]),\n        "sigma": float(run_config["sigma"]) if "sigma" in run_config else np.nan,\n        "epochs_completed": len(history["epoch"]),\n        "final_train_loss": history["train_loss"][-1] if history["train_loss"] else np.nan,\n        "final_train_acc": history["train_acc"][-1] if history["train_acc"] else np.nan,\n        "final_test_loss": history["test_loss"][-1] if history["test_loss"] else np.nan,\n        "final_test_acc": history["test_acc"][-1] if history["test_acc"] else np.nan,\n        "best_test_loss": best_test_loss,\n        "best_test_acc": best_test_acc,\n        "diverged": diverged,\n        "duration_sec": duration_sec,\n        "history": history,\n        "state_dict": state_dict_cpu,\n    }\n    del model\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n    return result\n\n\ndef train_backprop_checkpoint_states(data, dimensions, epochs, seed, device, lr, checkpoint_epochs, print_every_epoch=100):\n    checkpoint_epochs = sorted(set(int(epoch) for epoch in checkpoint_epochs))\n    set_seed(seed)\n    model = make_model(dimensions, require_grad=True, device=device)\n    optimizer = torch.optim.SGD(model.parameters(), lr=lr)\n    checkpoint_states = {}\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        for xb, yb, _ in data["train_loader"]:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            backprop_step(model, xb, yb, optimizer=optimizer)\n\n        if epoch in checkpoint_epochs:\n            checkpoint_states[epoch] = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n\n        if epoch % print_every_epoch == 0 or epoch == 1 or epoch == epochs or epoch in checkpoint_epochs:\n            train_loss, train_acc = evaluate_model(model, data["train_eval_loader"], device)\n            test_loss, test_acc = evaluate_model(model, data["test_loader"], device)\n            print(\n                f"    bp checkpoint training epoch {epoch:4d}/{epochs} | "\n                f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "\n                f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"\n            )\n\n    return checkpoint_states\n\n\ndef plot_layer_output_norm_histories(results_by_method):\n    first_result = next((results_by_method[method] for method in METHODS if method in results_by_method), None)\n    if first_result is None:\n        return\n\n    layer_output_history = first_result["history"].get("layer_output_norms", [])\n    if len(layer_output_history) == 0:\n        return\n\n    num_layers = len(layer_output_history[0])\n    fig, axes = plt.subplots(num_layers, 1, figsize=(10, 3.4 * num_layers), sharex=True, squeeze=False)\n    axes = axes.flatten()\n\n    for layer_idx in range(num_layers):\n        fan_in = int(first_result["state_dict"][f"layers.{layer_idx}.weight"].shape[1])\n        layer_label = "Input layer" if layer_idx == 0 else f"Hidden layer {layer_idx} output"\n        axis = axes[layer_idx]\n        for method in METHODS:\n            if method not in results_by_method:\n                continue\n            history = results_by_method[method]["history"]\n            values = [epoch_values[layer_idx] for epoch_values in history.get("layer_output_norms", [])]\n            if len(values) == 0:\n                continue\n            axis.plot(history["epoch"], values, label=METHOD_LABELS[method], color=METHOD_COLORS[method])\n        axis.set_title(f"{layer_label} norm / sqrt({fan_in} + 1)")\n        axis.set_ylabel("Average norm")\n        axis.legend(ncol=2)\n\n    axes[-1].set_xlabel("Epoch")\n    fig.tight_layout()\n    plt.show()\n\n\ndef layer_parameter_slices(model):\n    layer_slices = []\n    start = 0\n    for layer_idx, layer in enumerate(model.layers):\n        layer_parameter_count = layer.weight.numel() + layer.bias.numel()\n        layer_slices.append((layer_idx, slice(start, start + layer_parameter_count), f"Layer {layer_idx + 1}"))\n        start += layer_parameter_count\n    return layer_slices\n\n\ndef analyze_frozen_backprop_estimators(\n    checkpoint_states,\n    data,\n    dimensions,\n    device,\n    method_sigmas,\n    num_perturbations=50,\n    batch_size=128,\n):\n    analysis_dataset = TensorDataset(data["x_train"], data["y_train"])\n    analysis_loader = DataLoader(\n        analysis_dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=(device.type == "cuda"),\n    )\n\n    rows = []\n    perturbation_methods = list(method_sigmas.keys())\n\n    for checkpoint_epoch, state_dict in checkpoint_states.items():\n        model = make_model(dimensions, require_grad=True, device=device)\n        model.load_state_dict(state_dict)\n        component_specs = [(-1, slice(None), "All layers")] + layer_parameter_slices(model)\n\n        batch_metrics = {\n            method: {\n                layer_index: {\n                    "component": "all" if layer_index == -1 else f"layer_{layer_index + 1}",\n                    "component_label": component_label,\n                    "avg_sample_cosine": [],\n                    "mean_estimate_cosine": [],\n                    "sample_variance": [],\n                    "batch_variance": [],\n                }\n                for layer_index, _, component_label in component_specs\n            }\n            for method in perturbation_methods\n        }\n\n        for xb, yb in analysis_loader:\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            true_update = true_gradient_vector(model, xb, yb)\n\n            for method in perturbation_methods:\n                sigma = method_sigmas[method]\n                estimates = [\n                    perturbation_gradient_estimate_vector(method, model, xb, yb, sigma)\n                    for _ in range(num_perturbations)\n                ]\n                estimate_matrix = torch.stack(estimates, dim=0)\n                mean_estimate = estimate_matrix.mean(dim=0)\n\n                for layer_index, component_slice, _ in component_specs:\n                    component_true_update = true_update[component_slice]\n                    component_estimate_matrix = estimate_matrix[:, component_slice]\n                    component_mean_estimate = mean_estimate[component_slice]\n\n                    avg_sample_cosine = float(\n                        np.mean(\n                            [\n                                cosine_similarity_safe(component_estimate, component_true_update)\n                                for component_estimate in component_estimate_matrix\n                            ]\n                        )\n                    )\n                    mean_estimate_cosine = cosine_similarity_safe(component_mean_estimate, component_true_update)\n                    sample_variance, batch_variance = variance_against_true_gradient(\n                        component_estimate_matrix,\n                        component_true_update,\n                    )\n\n                    metrics = batch_metrics[method][layer_index]\n                    metrics["avg_sample_cosine"].append(avg_sample_cosine)\n                    metrics["mean_estimate_cosine"].append(mean_estimate_cosine)\n                    metrics["sample_variance"].append(sample_variance)\n                    metrics["batch_variance"].append(batch_variance)\n\n        checkpoint_fraction = checkpoint_epoch / max(checkpoint_states.keys())\n        for method in perturbation_methods:\n            for layer_index, _, component_label in component_specs:\n                metrics = batch_metrics[method][layer_index]\n                rows.append({\n                    "checkpoint_epoch": checkpoint_epoch,\n                    "checkpoint_fraction": checkpoint_fraction,\n                    "method": method,\n                    "sigma": method_sigmas[method],\n                    "component": metrics["component"],\n                    "component_label": component_label,\n                    "layer_index": layer_index,\n                    "avg_sample_cosine": float(np.mean(metrics["avg_sample_cosine"])),\n                    "mean_estimate_cosine": float(np.mean(metrics["mean_estimate_cosine"])),\n                    "sample_variance": float(np.mean(metrics["sample_variance"])),\n                    "batch_variance": float(np.mean(metrics["batch_variance"])),\n                })\n\n    return pd.DataFrame(rows)\n\n\ndef plot_frozen_estimator_statistics(stats_df):\n    metrics = [\n        ("avg_sample_cosine", "Average Sample Cosine", "Cosine"),\n        ("mean_estimate_cosine", "Cosine of Mean Estimator", "Cosine"),\n        ("sample_variance", "Per-Sample Variance to True Gradient", "Mean squared distance per parameter"),\n        ("batch_variance", "Batch-Average Variance to True Gradient", "Mean squared distance per parameter"),\n    ]\n    perturbation_methods = [method for method in METHODS if method in {"np", "np_fan_in", "np_fixed", "wp"}]\n\n    overall_df = stats_df[stats_df["layer_index"] == -1] if "layer_index" in stats_df.columns else stats_df\n    fig, axes = plt.subplots(1, len(metrics), figsize=(24, 4), sharex=True)\n\n    for method in perturbation_methods:\n        method_df = overall_df[overall_df["method"] == method].sort_values("checkpoint_epoch")\n        if len(method_df) == 0:\n            continue\n        x = method_df["checkpoint_epoch"].to_numpy()\n        for axis, (column, title, ylabel) in zip(axes, metrics):\n            axis.plot(x, method_df[column].to_numpy(), marker="o", color=METHOD_COLORS[method], label=METHOD_LABELS[method])\n            axis.set_title(f"All layers | {title}")\n            axis.set_xlabel("Checkpoint epoch")\n            axis.set_ylabel(ylabel)\n            axis.legend()\n\n    fig.tight_layout()\n    plt.show()\n\n    if "layer_index" not in stats_df.columns:\n        return\n\n    layer_indices = sorted(layer_index for layer_index in stats_df["layer_index"].unique() if layer_index >= 0)\n    if len(layer_indices) == 0:\n        return\n\n    fig, axes = plt.subplots(\n        len(layer_indices),\n        len(metrics),\n        figsize=(6 * len(metrics), 3.8 * len(layer_indices)),\n        sharex=True,\n        squeeze=False,\n    )\n\n    for row, layer_index in enumerate(layer_indices):\n        layer_df = stats_df[stats_df["layer_index"] == layer_index]\n        component_label = layer_df["component_label"].iloc[0]\n\n        for col, (column, title, ylabel) in enumerate(metrics):\n            axis = axes[row, col]\n            for method in perturbation_methods:\n                method_df = layer_df[layer_df["method"] == method].sort_values("checkpoint_epoch")\n                if len(method_df) == 0:\n                    continue\n                axis.plot(\n                    method_df["checkpoint_epoch"].to_numpy(),\n                    method_df[column].to_numpy(),\n                    marker="o",\n                    color=METHOD_COLORS[method],\n                    label=METHOD_LABELS[method],\n                )\n            axis.set_title(f"{component_label} | {title}")\n            axis.set_xlabel("Checkpoint epoch")\n            axis.set_ylabel(ylabel)\n            if col == len(metrics) - 1:\n                axis.legend(fontsize=8)\n\n    fig.tight_layout()\n    plt.show()\n\n\ndef analyze_frozen_backprop_sigma_grid(\n    checkpoint_states,\n    data,\n    dimensions,\n    device,\n    method_sigma_grid,\n    num_perturbations=50,\n    batch_size=128,\n):\n    analysis_dataset = TensorDataset(data["x_train"], data["y_train"])\n    analysis_loader = DataLoader(\n        analysis_dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=(device.type == "cuda"),\n    )\n\n    rows = []\n    methods = list(method_sigma_grid.keys())\n\n    for checkpoint_epoch, state_dict in checkpoint_states.items():\n        model = make_model(dimensions, require_grad=True, device=device)\n        model.load_state_dict(state_dict)\n\n        batch_metrics = {\n            (method, sigma): {\n                "avg_sample_cosine": [],\n                "mean_estimate_cosine": [],\n                "sample_variance": [],\n                "batch_variance": [],\n            }\n            for method in methods\n            for sigma in method_sigma_grid[method]\n        }\n\n        for batch in analysis_loader:\n            if len(batch) == 3:\n                xb, yb, _ = batch\n            else:\n                xb, yb = batch\n            xb = xb.to(device, non_blocking=True)\n            yb = yb.to(device, non_blocking=True)\n            true_update = true_gradient_vector(model, xb, yb)\n\n            for method in methods:\n                for sigma in method_sigma_grid[method]:\n                    estimates = [\n                        perturbation_gradient_estimate_vector(method, model, xb, yb, sigma)\n                        for _ in range(num_perturbations)\n                    ]\n                    estimate_matrix = torch.stack(estimates, dim=0)\n                    mean_estimate = estimate_matrix.mean(dim=0)\n\n                    avg_sample_cosine = float(\n                        sum(cosine_similarity_safe(estimate, true_update) for estimate in estimates) / num_perturbations\n                    )\n                    mean_estimate_cosine = cosine_similarity_safe(mean_estimate, true_update)\n                    sample_variance, batch_variance = variance_against_true_gradient(estimate_matrix, true_update)\n\n                    metrics = batch_metrics[(method, sigma)]\n                    metrics["avg_sample_cosine"].append(avg_sample_cosine)\n                    metrics["mean_estimate_cosine"].append(mean_estimate_cosine)\n                    metrics["sample_variance"].append(sample_variance)\n                    metrics["batch_variance"].append(batch_variance)\n\n        checkpoint_fraction = checkpoint_epoch / max(checkpoint_states.keys())\n        for method in methods:\n            for sigma in method_sigma_grid[method]:\n                metrics = batch_metrics[(method, sigma)]\n                rows.append({\n                    "checkpoint_epoch": checkpoint_epoch,\n                    "checkpoint_fraction": checkpoint_fraction,\n                    "method": method,\n                    "sigma": float(sigma),\n                    "avg_sample_cosine": float(np.mean(metrics["avg_sample_cosine"])),\n                    "mean_estimate_cosine": float(np.mean(metrics["mean_estimate_cosine"])),\n                    "sample_variance": float(np.mean(metrics["sample_variance"])),\n                    "batch_variance": float(np.mean(metrics["batch_variance"])),\n                })\n\n    return pd.DataFrame(rows)\n\n\ndef plot_frozen_sigma_search_results(stats_df):\n    metrics = [\n        ("avg_sample_cosine", "Average Sample Cosine", "Cosine"),\n        ("mean_estimate_cosine", "Cosine of Mean Estimator", "Cosine"),\n        ("sample_variance", "Per-Sample Variance to True Gradient", "Mean squared distance per parameter"),\n        ("batch_variance", "Batch-Average Variance to True Gradient", "Mean squared distance per parameter"),\n    ]\n    checkpoint_epochs = sorted(stats_df["checkpoint_epoch"].unique())\n\n    for method in [method for method in METHODS if method in set(stats_df["method"])] :\n        method_df = stats_df[stats_df["method"] == method]\n        fig, axes = plt.subplots(\n            len(checkpoint_epochs),\n            len(metrics),\n            figsize=(5 * len(metrics), 3.8 * len(checkpoint_epochs)),\n            squeeze=False,\n        )\n\n        for row, checkpoint_epoch in enumerate(checkpoint_epochs):\n            subset = method_df[method_df["checkpoint_epoch"] == checkpoint_epoch].sort_values("sigma")\n            sigma_labels = [f"{sigma:.4g}" for sigma in subset["sigma"].to_numpy()]\n\n            for col, (column, title, ylabel) in enumerate(metrics):\n                axis = axes[row, col]\n                axis.bar(sigma_labels, subset[column].to_numpy(), color=METHOD_COLORS[method])\n                axis.set_title(f"epoch {checkpoint_epoch} | {title}")\n                axis.set_xlabel("sigma")\n                axis.set_ylabel(ylabel)\n                axis.tick_params(axis="x", rotation=45)\n\n        fig.suptitle(f"{METHOD_LABELS[method]} sigma search", y=1.02)\n        fig.tight_layout()\n        plt.show()\n\n\ndef build_grid(space):\n    keys = list(space.keys())\n    values = [space[key] for key in keys]\n    return [dict(zip(keys, combo)) for combo in itertools.product(*values)]\n\n\ndef run_sweep(search_spaces, data, dimensions, sweep_epochs, seeds, device):\n    results = []\n    flat_records = []\n    total_runs = sum(len(configs) * len(seeds) for configs in search_spaces.values())\n    run_idx = 0\n\n    print(f"Starting sweep with total_runs={total_runs}, device={device}")\n    for method in METHODS:\n        configs = search_spaces[method]\n        for config in configs:\n            for seed in seeds:\n                run_idx += 1\n                sigma_str = f", sigma={config[\'sigma\']:.4g}" if "sigma" in config else ""\n                print(f"\\n=== Run {run_idx}/{total_runs} | {method} | lr={config[\'lr\']:.4g}{sigma_str} | seed={seed} ===")\n                result = train_one_run(\n                    method=method,\n                    run_config=config,\n                    data=data,\n                    dimensions=dimensions,\n                    epochs=sweep_epochs,\n                    seed=seed,\n                    device=device,\n                    print_every_epoch=1,\n                )\n                results.append(result)\n                flat_record = {k: v for k, v in result.items() if k not in {"history", "state_dict"}}\n                flat_record["seed"] = seed\n                flat_records.append(flat_record)\n\n    df = pd.DataFrame(flat_records)\n    if len(df) > 0:\n        df = df.sort_values(\n            ["method", "final_test_acc", "best_test_acc", "final_test_loss"],\n            ascending=[True, False, False, True],\n        ).reset_index(drop=True)\n    return results, df\n\n\ndef summarize_top_configs(df, top_k=5):\n    if len(df) == 0:\n        return df\n    frames = []\n    for method, group in df.groupby("method"):\n        frames.append(group.head(top_k))\n    return pd.concat(frames, ignore_index=True)\n\n\ndef select_best_configs(df):\n    rows = []\n    good = df[df["diverged"] == False]\n    for method, group in good.groupby("method"):\n        best = group.sort_values(\n            ["final_test_acc", "best_test_acc", "final_test_loss"],\n            ascending=[False, False, True],\n        ).iloc[0]\n        rows.append(best)\n    return pd.DataFrame(rows).reset_index(drop=True)\n\n\ndef run_best_config_comparison(best_df, data, dimensions, final_epochs, seed, device):\n    comparison_results = {}\n    for row in best_df.itertuples(index=False):\n        config = {"lr": float(row.lr)}\n        if not pd.isna(row.sigma):\n            config["sigma"] = float(row.sigma)\n        print(\n            f"\\n### Final run | {row.method} | lr={config[\'lr\']:.4g}" +\n            (f", sigma={config[\'sigma\']:.4g}" if \'sigma\' in config else "")\n        )\n        comparison_results[row.method] = train_one_run(\n            method=row.method,\n            run_config=config,\n            data=data,\n            dimensions=dimensions,\n            epochs=final_epochs,\n            seed=seed,\n            device=device,\n            print_every_epoch=1,\n        )\n    return comparison_results\n\n\ndef results_table(results_by_method):\n    rows = []\n    for method in METHODS:\n        if method not in results_by_method:\n            continue\n        result = results_by_method[method]\n        rows.append(\n            {\n                "method": method,\n                "final_train_loss": result["final_train_loss"],\n                "final_train_acc": result["final_train_acc"],\n                "final_test_loss": result["final_test_loss"],\n                "final_test_acc": result["final_test_acc"],\n                "best_test_loss": result["best_test_loss"],\n                "best_test_acc": result["best_test_acc"],\n                "diverged": result["diverged"],\n                "duration_sec": result["duration_sec"],\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef plot_training_histories(results_by_method):\n    fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)\n    for method in METHODS:\n        if method not in results_by_method:\n            continue\n        result = results_by_method[method]\n        history = result["history"]\n        axes[0].plot(history["epoch"], history["train_loss"], label=f"{METHOD_LABELS[method]} train", color=METHOD_COLORS[method])\n        axes[0].plot(history["epoch"], history["test_loss"], linestyle="--", label=f"{METHOD_LABELS[method]} test", color=METHOD_COLORS[method])\n        axes[1].plot(history["epoch"], history["train_acc"], label=f"{METHOD_LABELS[method]} train", color=METHOD_COLORS[method])\n        axes[1].plot(history["epoch"], history["test_acc"], linestyle="--", label=f"{METHOD_LABELS[method]} test", color=METHOD_COLORS[method])\n        axes[2].plot(history["epoch"], history["weight_norm"], label=METHOD_LABELS[method], color=METHOD_COLORS[method])\n    axes[0].set_title("Loss")\n    axes[0].set_ylabel("MSE")\n    axes[0].legend(ncol=2)\n    axes[1].set_title("Accuracy")\n    axes[1].set_ylabel("Accuracy")\n    axes[1].set_ylim(0.0, 1.0)\n    axes[1].legend(ncol=2)\n    axes[2].set_title("Weight Norm")\n    axes[2].set_ylabel("L2 norm")\n    axes[2].set_xlabel("Epoch")\n    axes[2].legend(ncol=2)\n    fig.tight_layout()\n    plt.show()\n',
}


_SOURCE_API_CACHE = {}


def load_embedded_task_api(task_key: str) -> dict:
    """Execute one embedded task implementation in its own namespace."""
    if task_key in _SOURCE_API_CACHE:
        return _SOURCE_API_CACHE[task_key]
    namespace = {"__name__": f"_{task_key}_embedded_api"}
    exec(compile(EMBEDDED_TASK_SOURCE_CODE[task_key], f"{task_key}_embedded_api", "exec"), namespace)
    _SOURCE_API_CACHE[task_key] = namespace
    return namespace


In [ ]:

TASK_CONFIGS = {
    "sinus": {
        "display_name": "Sinus",
        "task_type": "regression",
        "data_loader": "load_synthetic_vector_regression_data",
        "data_kwargs": {
            "input_dim": 8,
            "input_scale": 1.0,
            "n_train": 512,
            "n_test": 512,
            "noise_std": 0.1,
            "train_eval_limit": None,
            "batch_size": 64,
            "eval_batch_size": 512,
            "seed": 0,
            "num_workers": 0,
        },
        "dimensions": (8, 8, 8, 1),
        "methods": ["bp", "np", "np_fixed", "np_fan_in", "wp"],
        "run_epochs": 500,
        "run_seed": 1,
        "run_print_every_epoch": 1,
        "run_configs": {
            "bp": {"lr": 0.1},
            "np": {"lr": 0.2 * 0.75, "sigma": 0.15},
            "np_fan_in": {"lr": 0.15 * 0.75, "sigma": 0.1},
            "np_fixed": {"lr": 0.11 * 0.75, "sigma": 0.3},
            "wp": {"lr": 0.09 * 0.75, "sigma": 0.225},
        },
        "analysis": {
            "epochs": 240,
            "seed": 0,
            "bp_lr": 0.1,
            "num_perturbations": 240,
            "batch_size": 64,
            "checkpoint_epochs": [1, 120, 240],
            "method_sigmas": {
                "np": 0.15,
                "np_fan_in": 0.1,
                "np_fixed": 0.3,
                "wp": 0.225,
            },
        },
    },
    "california_housing": {
        "display_name": "California Housing",
        "task_type": "regression",
        "data_loader": "load_california_housing",
        "data_kwargs": {
            "test_size": 0.2,
            "batch_size": 256,
            "eval_batch_size": 4096,
            "seed": 0,
            "num_workers": 0,
            "data_home": str(DATA_DIR / "sklearn"),
        },
        "dimensions": (8, 128, 64, 1),
        "methods": METHODS,
        "run_epochs": 200,
        "run_seed": 0,
        "run_print_every_epoch": 1,
        "run_configs": {
            "bp": {"lr": 0.1},
            "np": {"lr": 0.03 * 0.9, "sigma": 0.1},
            "np_fan_in": {"lr": 0.025 * 0.7, "sigma": 0.0525},
            "np_fixed": {"lr": 0.025 * 0.7, "sigma": 0.45},
            "wp": {"lr": 0.015 * 0.9, "sigma": 0.1},
        },
        "analysis": {
            "epochs": 50,
            "seed": 0,
            "bp_lr": 0.01,
            "num_perturbations": 200,
            "batch_size": 256,
            "checkpoint_epochs": [1, 25, 50],
            "method_sigmas": {
                "np": 0.1,
                "np_fan_in": 0.0525,
                "np_fixed": 0.45,
                "wp": 0.1,
            },
        },
    },
    "mnist": {
        "display_name": "MNIST",
        "task_type": "classification",
        "data_loader": "load_mnist",
        "data_kwargs": {
            "train_limit": 20000,
            "test_limit": 5000,
            "train_eval_limit": 4000,
            "batch_size": 128,
            "eval_batch_size": 1024,
            "data_dir": str(DATA_DIR / "torchvision"),
            "seed": 0,
            "mean_center_only": True,
            "num_workers": 0,
        },
        "dimensions": (28 * 28, 256, 128, 10),
        "methods": METHODS,
        "run_epochs": 20 * 20,
        "run_seed": 0,
        "run_print_every_epoch": 5,
        "run_configs": {
            "bp": {"lr": 0.5},
            "np": {"lr": 0.12 * 0.425, "sigma": 0.0325},
            "np_fan_in": {"lr": 0.0325 * 0.375, "sigma": 0.00875},
            "np_fixed": {"lr": 0.0225 * 0.25, "sigma": 0.12},
            "wp": {"lr": 0.015 * 0.25, "sigma": 0.0275},
        },
        "analysis": {
            "epochs": 20,
            "seed": 0,
            "bp_lr": 0.2,
            "num_perturbations": 100,
            "batch_size": 128,
            "checkpoint_epochs": [1, 10, 20],
            "method_sigmas": {
                "np": 0.0325,
                "np_fan_in": 0.00875,
                "np_fixed": 0.12,
                "wp": 0.0275,
            },
        },
    },
    "mnist_scaled_input_training": {
        "display_name": "MNIST x5",
        "task_type": "classification",
        "source_task_key": "mnist",
        "input_scale": 5.0,
        "reuse_diagnostic_task": "mnist_scaled_input",
        "data_loader": "load_mnist",
        "data_kwargs": {
            "train_limit": 20000,
            "test_limit": 5000,
            "train_eval_limit": 4000,
            "batch_size": 128,
            "eval_batch_size": 1024,
            "data_dir": str(DATA_DIR / "torchvision"),
            "seed": 0,
            "mean_center_only": True,
            "num_workers": 0,
        },
        "dimensions": (28 * 28, 256, 128, 10),
        "methods": METHODS,
        "run_epochs": 20 * 20,
        "run_seed": 0,
        "run_print_every_epoch": 5,
        "run_configs": {
            "bp": {"lr": 0.5},
            "np": {"lr": 0.12 * 0.425, "sigma": 0.0325},
            "np_fan_in": {"lr": 0.0325 * 0.375, "sigma": 0.00875},
            "np_fixed": {"lr": 0.0225 * 0.25, "sigma": 0.12},
            "wp": {"lr": 0.015 * 0.25, "sigma": 0.0275},
        },
        "analysis": {
            "epochs": 20,
            "seed": 0,
            "bp_lr": 0.2,
            "num_perturbations": 100,
            "batch_size": 128,
            "checkpoint_epochs": [1, 10, 20],
            "method_sigmas": {
                "np": 0.0325,
                "np_fan_in": 0.00875,
                "np_fixed": 0.12,
                "wp": 0.0275,
            },
        },
    },
    "cifar10": {
        "display_name": "CIFAR-10",
        "task_type": "classification",
        "data_loader": "load_cifar10",
        "data_kwargs": {
            "train_limit": 20000,
            "test_limit": 5000,
            "train_eval_limit": 4000,
            "batch_size": 128,
            "eval_batch_size": 1024,
            "data_dir": str(DATA_DIR / "torchvision"),
            "seed": 0,
            "mean_center_only": True,
            "num_workers": 0,
        },
        "dimensions": (32 * 32 * 3, 512, 256, 10),
        "methods": METHODS,
        "run_epochs": 80,
        "run_seed": 0,
        "run_print_every_epoch": 1,
        "run_configs": {
            "bp": {"lr": 0.1},
            "np": {"lr": (0.18 / 6) * 0.4, "sigma": 0.0175},
            "np_fan_in": {"lr": (0.15 / 9) * 0.4, "sigma": 0.0055},
            "np_fixed": {"lr": (0.09 / 9) * 0.4, "sigma": 0.19},
            "wp": {"lr": (0.036 / 9) * 0.4, "sigma": 0.0125},
        },
        "analysis": {
            "epochs": 50,
            "seed": 0,
            "bp_lr": 0.1,
            "num_perturbations": 10,
            "batch_size": 128,
            "checkpoint_epochs": [1, 25, 50],
            "method_sigmas": {
                "np": 0.0175,
                "np_fan_in": 0.0055,
                "np_fixed": 0.19,
                "wp": 0.0125,
            },
        },
    },
}

DIAGNOSTIC_TASK_CONFIGS = {
    "sinus_scaled_input": {
        "display_name": "Sinus [-5, 5]",
        "source_task_key": "sinus",
        "data_loader": "load_synthetic_vector_regression_data",
        "data_kwargs": {
            "input_dim": 8,
            "input_scale": 5.0,
            "n_train": 512,
            "n_test": 512,
            "noise_std": 0.1,
            "train_eval_limit": None,
            "batch_size": 64,
            "eval_batch_size": 512,
            "seed": 0,
            "num_workers": 0,
        },
        "dimensions": (8, 8, 8, 1),
        "analysis": {
            "epochs": 240,
            "seed": 0,
            "bp_lr": 0.1,
            "num_perturbations": 240,
            "batch_size": 64,
            "checkpoint_epochs": [1, 120, 240],
            "method_sigmas": {
                "np": 0.15,
                "np_fan_in": 0.1,
                "np_fixed": 0.3,
                "wp": 0.225,
            },
        },
    },
    "mnist_scaled_input": {
        "display_name": "MNIST x5",
        "source_task_key": "mnist",
        "input_scale": 5.0,
        "data_loader": "load_mnist",
        "data_kwargs": {
            "train_limit": 20000,
            "test_limit": 5000,
            "train_eval_limit": 4000,
            "batch_size": 128,
            "eval_batch_size": 1024,
            "data_dir": str(DATA_DIR / "torchvision"),
            "seed": 0,
            "mean_center_only": True,
            "num_workers": 0,
        },
        "dimensions": (28 * 28, 256, 128, 10),
        "analysis": {
            "epochs": 20,
            "seed": 0,
            "bp_lr": 0.2,
            "num_perturbations": 100,
            "batch_size": 128,
            "checkpoint_epochs": [1, 10, 20],
            "method_sigmas": {
                "np": 0.0325,
                "np_fan_in": 0.00875,
                "np_fixed": 0.12,
                "wp": 0.0275,
            },
        },
    },
}


pd.DataFrame(
    [
        {
            "task": config["display_name"],
            "run_epochs": config["run_epochs"],
            "run_seed": config["run_seed"],
            "analysis_epochs": config["analysis"]["epochs"],
            "analysis_checkpoints": config["analysis"]["checkpoint_epochs"],
            "num_perturbations": config["analysis"]["num_perturbations"],
        }
        for config in TASK_CONFIGS.values()
    ]
)


In [ ]:

def task_cache_path(task_key: str) -> Path:
    return CACHE_DIR / f"{task_key}_results.pkl"


def save_pdf(fig, filename: str) -> Path:
    path = FIGURE_DIR / filename
    fig.savefig(path, format="pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return path


def add_curve_legend(ax, methods):
    handles = []
    for method in methods:
        handles.append(
            Line2D([0], [0], color=METHOD_COLORS[method], lw=LEGEND_LINE_WIDTH, label=METHOD_LABELS[method])
        )
    handles.extend(
        [
            Line2D([0], [0], color="black", lw=LEGEND_LINE_WIDTH, linestyle="-", label="Training"),
            Line2D([0], [0], color="black", lw=LEGEND_LINE_WIDTH, linestyle="--", label="Test"),
        ]
    )
    ax.legend(handles=handles, ncol=2, frameon=True, framealpha=0.95)


def plot_loss_curves(results_by_method, task_key: str, methods=None):
    methods = methods or METHODS
    fig, ax = plt.subplots(figsize=(5.8, 3.6))
    for method in methods:
        if method not in results_by_method:
            continue
        history = results_by_method[method]["history"]
        ax.plot(history["epoch"], history["train_loss"], color=METHOD_COLORS[method], lw=CURVE_LINE_WIDTH, linestyle="-")
        ax.plot(history["epoch"], history["test_loss"], color=METHOD_COLORS[method], lw=CURVE_LINE_WIDTH, linestyle="--")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE loss")
    add_curve_legend(ax, [method for method in methods if method in results_by_method])
    fig.tight_layout()
    return save_pdf(fig, f"{task_key}_loss.pdf")


def plot_accuracy_curves(results_by_method, task_key: str, methods=None):
    methods = methods or METHODS
    fig, ax = plt.subplots(figsize=(5.8, 3.6))
    for method in methods:
        if method not in results_by_method:
            continue
        history = results_by_method[method]["history"]
        ax.plot(history["epoch"], history["train_acc"], color=METHOD_COLORS[method], lw=CURVE_LINE_WIDTH, linestyle="-")
        ax.plot(history["epoch"], history["test_acc"], color=METHOD_COLORS[method], lw=CURVE_LINE_WIDTH, linestyle="--")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0.0, 1.0)
    add_curve_legend(ax, [method for method in methods if method in results_by_method])
    fig.tight_layout()
    return save_pdf(fig, f"{task_key}_accuracy.pdf")


def summarize_frozen_statistics(frozen_stats_df, cosine_column=COSINE_COLUMN, variance_column=VARIANCE_COLUMN):
    all_layers = frozen_stats_df[frozen_stats_df["component"] == "all"].copy()
    return (
        all_layers.groupby("method", as_index=False)
        .agg(
            cosine_similarity=(cosine_column, "mean"),
            update_variance=(variance_column, "mean"),
            sigma=("sigma", "mean"),
        )
        .sort_values("method")
    )


def should_use_log_scale(values, ratio_threshold=100.0):
    positive_values = [float(value) for value in values if pd.notna(value) and float(value) > 0.0]
    if len(positive_values) < 2:
        return False
    return max(positive_values) / min(positive_values) >= ratio_threshold


def cosine_axis_limits(values, padding_fraction=0.15):
    finite_values = np.asarray([float(value) for value in values if pd.notna(value) and np.isfinite(float(value))])
    if finite_values.size == 0:
        return 0.0, 1.0

    min_value = float(finite_values.min())
    max_value = float(finite_values.max())

    if min_value >= 0.0:
        upper = max_value * (1.0 + padding_fraction) if max_value > 0.0 else 0.05
        return 0.0, min(1.0, upper)
    if max_value <= 0.0:
        lower = min_value * (1.0 + padding_fraction)
        return max(-1.0, lower), 0.0

    value_range = max_value - min_value
    padding = value_range * padding_fraction if value_range > 0.0 else 0.05
    lower = max(-1.0, min_value - padding)
    upper = min(1.0, max_value + padding)
    return lower, upper


def plot_diagnostic_bar(summary_df, task_key: str, value_column: str, ylabel: str, filename_suffix: str):
    plot_df = summary_df.set_index("method").loc[PERTURBATION_METHODS].reset_index()
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    x = np.arange(len(plot_df))
    values = plot_df[value_column].astype(float).to_numpy()
    use_log_scale = value_column == "update_variance" and should_use_log_scale(values)

    for index, row in plot_df.iterrows():
        method = row["method"]
        ax.bar(x[index], row[value_column], color=METHOD_COLORS[method], width=0.72, label=METHOD_LABELS[method])

    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS[method] for method in plot_df["method"]], rotation=20, ha="right")
    ax.set_ylabel(f"{ylabel} (log scale)" if use_log_scale else ylabel)
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True, framealpha=0.95)

    if use_log_scale:
        positive_values = values[values > 0.0]
        ax.set_yscale("log")
        ax.set_ylim(float(positive_values.min()) / 3.0, float(positive_values.max()) * 3.0)
        ax.grid(True, which="both", axis="y", alpha=0.25)
    elif value_column == "cosine_similarity":
        ax.set_ylim(*cosine_axis_limits(values))

    fig.tight_layout()
    return save_pdf(fig, f"{task_key}_{filename_suffix}.pdf")


def plot_variance_bar(summary_df, task_key: str):
    return plot_diagnostic_bar(summary_df, task_key, "update_variance", "Update variance", "variance")


def plot_cosine_bar(summary_df, task_key: str):
    return plot_diagnostic_bar(summary_df, task_key, "cosine_similarity", "Cosine similarity", "cosine")


def plot_scaled_diagnostic_comparison(
    base_summary_df,
    scaled_summary_df,
    task_key: str,
    value_column: str,
    ylabel: str,
    filename_suffix: str,
    base_label: str,
    scaled_label: str,
):
    base_df = base_summary_df.set_index("method")
    scaled_df = scaled_summary_df.set_index("method")
    methods = [method for method in PERTURBATION_METHODS if method in base_df.index and method in scaled_df.index]

    base_values = np.array([float(base_df.loc[method, value_column]) for method in methods])
    scaled_values = np.array([float(scaled_df.loc[method, value_column]) for method in methods])
    all_values = np.concatenate([base_values, scaled_values])

    fig, ax = plt.subplots(figsize=(5.8, 3.4))
    x = np.arange(len(methods))
    width = 0.34

    for index, method in enumerate(methods):
        color = METHOD_COLORS[method]
        ax.bar(
            x[index] - width / 2,
            base_values[index],
            width=width,
            color=color,
            edgecolor="black",
            linewidth=0.5,
        )
        ax.bar(
            x[index] + width / 2,
            scaled_values[index],
            width=width,
            color=color,
            edgecolor="black",
            linewidth=0.5,
            hatch="////",
            alpha=0.78,
        )

    use_log_scale = value_column == "update_variance" and should_use_log_scale(all_values)
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS[method] for method in methods], rotation=20, ha="right")
    ax.set_ylabel(f"{ylabel} (log scale)" if use_log_scale else ylabel)

    condition_handles = [
        Patch(facecolor="0.6", edgecolor="black", label=base_label),
        Patch(facecolor="0.6", edgecolor="black", hatch="////", alpha=0.78, label=scaled_label),
    ]
    ax.legend(handles=condition_handles, frameon=True, framealpha=0.95)

    if use_log_scale:
        positive_values = all_values[all_values > 0.0]
        ax.set_yscale("log")
        ax.set_ylim(float(positive_values.min()) / 3.0, float(positive_values.max()) * 3.0)
        ax.grid(True, which="both", axis="y", alpha=0.25)
    elif value_column == "cosine_similarity":
        ax.set_ylim(*cosine_axis_limits(all_values))

    fig.tight_layout()
    return save_pdf(fig, f"{task_key}_{filename_suffix}_comparison.pdf")


def plot_scaled_cosine_comparison(base_summary_df, scaled_summary_df, task_key: str, base_label: str, scaled_label: str):
    return plot_scaled_diagnostic_comparison(
        base_summary_df,
        scaled_summary_df,
        task_key,
        "cosine_similarity",
        "Cosine similarity",
        "cosine",
        base_label,
        scaled_label,
    )


def plot_scaled_variance_comparison(base_summary_df, scaled_summary_df, task_key: str, base_label: str, scaled_label: str):
    return plot_scaled_diagnostic_comparison(
        base_summary_df,
        scaled_summary_df,
        task_key,
        "update_variance",
        "Update variance",
        "variance",
        base_label,
        scaled_label,
    )


def plot_scaled_diagnostic_comparisons(
    base_outputs,
    scaled_outputs,
    task_key: str,
    base_label: str = "Unscaled",
    scaled_label: str = "Scaled",
):
    figure_paths = [
        plot_scaled_variance_comparison(
            base_outputs["diagnostic_summary"],
            scaled_outputs["diagnostic_summary"],
            task_key,
            base_label,
            scaled_label,
        ),
        plot_scaled_cosine_comparison(
            base_outputs["diagnostic_summary"],
            scaled_outputs["diagnostic_summary"],
            task_key,
            base_label,
            scaled_label,
        ),
    ]
    print("Wrote comparison figures:")
    for path in figure_paths:
        print(f"  {path}")
    return figure_paths


def train_selected_configs(api, task_config):
    results_by_method = {}
    for method in task_config["methods"]:
        if method not in task_config["run_configs"]:
            continue
        config = task_config["run_configs"][method]
        sigma_str = f", sigma={config['sigma']:.4g}" if "sigma" in config else ""
        print(f"\n### Final run | {task_config['display_name']} | {method} | lr={config['lr']:.4g}{sigma_str}")
        results_by_method[method] = api["train_one_run"](
            method=method,
            run_config=config,
            data=task_config["data"],
            dimensions=task_config["dimensions"],
            epochs=task_config["run_epochs"],
            seed=task_config["run_seed"],
            device=api["device"],
            print_every_epoch=task_config["run_print_every_epoch"],
        )
    return results_by_method


def analyze_task_estimators(api, task_config):
    analysis = task_config["analysis"]
    print(
        f"\n### Frozen estimator analysis | {task_config['display_name']} | "
        f"epochs={analysis['epochs']}, seed={analysis['seed']}, bp_lr={analysis['bp_lr']}, "
        f"checkpoints={analysis['checkpoint_epochs']}, "
        f"num_perturbations={analysis['num_perturbations']}, batch_size={analysis['batch_size']}"
    )
    checkpoint_states = api["train_backprop_checkpoint_states"](
        data=task_config["data"],
        dimensions=task_config["dimensions"],
        epochs=analysis["epochs"],
        seed=analysis["seed"],
        device=api["device"],
        lr=analysis["bp_lr"],
        checkpoint_epochs=analysis["checkpoint_epochs"],
        print_every_epoch=max(1, analysis["epochs"] // 10),
    )
    frozen_stats_df = api["analyze_frozen_backprop_estimators"](
        checkpoint_states=checkpoint_states,
        data=task_config["data"],
        dimensions=task_config["dimensions"],
        device=api["device"],
        method_sigmas=analysis["method_sigmas"],
        num_perturbations=analysis["num_perturbations"],
        batch_size=analysis["batch_size"],
    )
    return checkpoint_states, frozen_stats_df


def run_task(task_key: str, force_recompute: bool = FORCE_RECOMPUTE):
    task_config = copy.deepcopy(TASK_CONFIGS[task_key])
    cache_path = task_cache_path(task_key)

    if USE_CACHE and cache_path.exists() and not force_recompute:
        print(f"Loading cached results for {task_config['display_name']}: {cache_path}")
        with cache_path.open("rb") as handle:
            outputs = pickle.load(handle)
        outputs["diagnostic_summary"] = summarize_frozen_statistics(outputs["frozen_stats_df"])
    else:
        api = load_embedded_task_api(task_config.get("source_task_key", task_key))
        data_loader = api[task_config["data_loader"]]
        data = data_loader(**task_config["data_kwargs"])
        data = scale_input_tensors(data, task_config.get("input_scale", 1.0))
        task_config["data"] = data
        print(
            f"Running {task_config['display_name']} | "
            f"epochs={task_config['run_epochs']}, seed={task_config['run_seed']}, "
            f"input_scale={task_config.get('input_scale', 1.0)}, "
            f"dims={task_config['dimensions']}, device={api['device']}"
        )
        results_by_method = train_selected_configs(api, task_config)

        reuse_diagnostic_task = task_config.get("reuse_diagnostic_task")
        if reuse_diagnostic_task is not None:
            diagnostic_output = globals().get("diagnostic_outputs", {}).get(reuse_diagnostic_task)
            if diagnostic_output is None and diagnostic_cache_path(reuse_diagnostic_task).exists():
                with diagnostic_cache_path(reuse_diagnostic_task).open("rb") as handle:
                    diagnostic_output = pickle.load(handle)
            if diagnostic_output is None:
                diagnostic_output = run_diagnostic_task(reuse_diagnostic_task)
            frozen_stats_df = diagnostic_output["frozen_stats_df"]
            diagnostic_summary = summarize_frozen_statistics(frozen_stats_df)
        else:
            _, frozen_stats_df = analyze_task_estimators(api, task_config)
            diagnostic_summary = summarize_frozen_statistics(frozen_stats_df)
        outputs = {
            "task_key": task_key,
            "task_config": {key: value for key, value in task_config.items() if key != "data"},
            "results_by_method": results_by_method,
            "frozen_stats_df": frozen_stats_df,
            "diagnostic_summary": diagnostic_summary,
        }
        if USE_CACHE:
            with cache_path.open("wb") as handle:
                pickle.dump(outputs, handle)
            print(f"Cached results to {cache_path}")

    figure_paths = []
    figure_paths.append(plot_loss_curves(outputs["results_by_method"], task_key, TASK_CONFIGS[task_key]["methods"]))
    if TASK_CONFIGS[task_key]["task_type"] == "classification":
        figure_paths.append(plot_accuracy_curves(outputs["results_by_method"], task_key, TASK_CONFIGS[task_key]["methods"]))
    figure_paths.append(plot_variance_bar(outputs["diagnostic_summary"], task_key))
    figure_paths.append(plot_cosine_bar(outputs["diagnostic_summary"], task_key))
    outputs["figure_paths"] = figure_paths

    print("Wrote figures:")
    for path in figure_paths:
        print(f"  {path}")
    return outputs



def diagnostic_cache_path(task_key: str) -> Path:
    task_config = DIAGNOSTIC_TASK_CONFIGS.get(task_key, {})
    input_scale = float(task_config.get("input_scale", 1.0))
    scale_suffix = f"_x{input_scale:g}" if input_scale != 1.0 else ""
    return CACHE_DIR / f"{task_key}{scale_suffix}_diagnostics.pkl"


def scale_input_tensors(data, scale):
    if scale is None or scale == 1.0:
        return data
    scaled_data = dict(data)
    scale = float(scale)
    for key in ["x_train", "x_test", "x_train_eval"]:
        if key in scaled_data:
            scaled_data[key].mul_(scale)
    return scaled_data


def run_diagnostic_task(task_key: str, force_recompute: bool = FORCE_RECOMPUTE):
    task_config = copy.deepcopy(DIAGNOSTIC_TASK_CONFIGS[task_key])
    cache_path = diagnostic_cache_path(task_key)

    if USE_CACHE and cache_path.exists() and not force_recompute:
        print(f"Loading cached diagnostics for {task_config['display_name']}: {cache_path}")
        with cache_path.open("rb") as handle:
            outputs = pickle.load(handle)
        outputs["diagnostic_summary"] = summarize_frozen_statistics(outputs["frozen_stats_df"])
    else:
        api = load_embedded_task_api(task_config["source_task_key"])
        data_loader = api[task_config["data_loader"]]
        data = data_loader(**task_config["data_kwargs"])
        data = scale_input_tensors(data, task_config.get("input_scale", 1.0))
        task_config["data"] = data
        analysis = task_config["analysis"]
        print(
            f"Running diagnostics only | {task_config['display_name']} | "
            f"epochs={analysis['epochs']}, checkpoints={analysis['checkpoint_epochs']}, "
            f"num_perturbations={analysis['num_perturbations']}, "
            f"input_scale={task_config.get('input_scale', 1.0)}, "
            f"dims={task_config['dimensions']}, device={api['device']}"
        )
        _, frozen_stats_df = analyze_task_estimators(api, task_config)
        diagnostic_summary = summarize_frozen_statistics(frozen_stats_df)
        outputs = {
            "task_key": task_key,
            "task_config": {key: value for key, value in task_config.items() if key != "data"},
            "frozen_stats_df": frozen_stats_df,
            "diagnostic_summary": diagnostic_summary,
        }
        if USE_CACHE:
            with cache_path.open("wb") as handle:
                pickle.dump(outputs, handle)
            print(f"Cached diagnostics to {cache_path}")

    figure_paths = [
        plot_variance_bar(outputs["diagnostic_summary"], task_key),
        plot_cosine_bar(outputs["diagnostic_summary"], task_key),
    ]
    outputs["figure_paths"] = figure_paths

    print("Wrote diagnostic figures:")
    for path in figure_paths:
        print(f"  {path}")
    return outputs


## Sinus

In [ ]:
all_task_outputs = globals().get("all_task_outputs", {})
all_task_outputs['sinus'] = run_task('sinus')
all_task_outputs['sinus']["diagnostic_summary"]


## Sinus [-5, 5] Diagnostics

In [ ]:
diagnostic_outputs = globals().get("diagnostic_outputs", {})
diagnostic_outputs["sinus_scaled_input"] = run_diagnostic_task("sinus_scaled_input")
diagnostic_outputs["sinus_scaled_input"]["diagnostic_summary"]


## Sinus Scale Comparison

In [ ]:
scale_comparison_outputs = globals().get("scale_comparison_outputs", {})
scale_comparison_outputs["sinus_scaled_input"] = plot_scaled_diagnostic_comparisons(
    all_task_outputs["sinus"],
    diagnostic_outputs["sinus_scaled_input"],
    task_key="sinus_scaled_input",
    base_label="[-1, 1]",
    scaled_label="[-5, 5]",
)


## California Housing

In [ ]:
all_task_outputs = globals().get("all_task_outputs", {})
all_task_outputs['california_housing'] = run_task('california_housing')
all_task_outputs['california_housing']["diagnostic_summary"]


## MNIST x5 Diagnostics

In [ ]:
diagnostic_outputs = globals().get("diagnostic_outputs", {})
diagnostic_outputs["mnist_scaled_input"] = run_diagnostic_task("mnist_scaled_input")
diagnostic_outputs["mnist_scaled_input"]["diagnostic_summary"]


## MNIST

In [ ]:
all_task_outputs = globals().get("all_task_outputs", {})
all_task_outputs['mnist'] = run_task('mnist')
all_task_outputs['mnist']["diagnostic_summary"]


## MNIST x5 Training

In [ ]:
all_task_outputs = globals().get("all_task_outputs", {})
all_task_outputs["mnist_scaled_input_training"] = run_task("mnist_scaled_input_training")
all_task_outputs["mnist_scaled_input_training"]["diagnostic_summary"]


## MNIST Scale Comparison

In [ ]:
scale_comparison_outputs = globals().get("scale_comparison_outputs", {})
scale_comparison_outputs["mnist_scaled_input"] = plot_scaled_diagnostic_comparisons(
    all_task_outputs["mnist"],
    all_task_outputs["mnist_scaled_input_training"],
    task_key="mnist_scaled_input",
    base_label="Original",
    scaled_label="x5",
)


## CIFAR-10

In [ ]:
all_task_outputs = globals().get("all_task_outputs", {})
all_task_outputs['cifar10'] = run_task('cifar10')
all_task_outputs['cifar10']["diagnostic_summary"]


## Summary Table

In [ ]:

def _best_history_value(history, key, reducer, default=np.nan):
    values = history.get(key, [])
    finite_values = [value for value in values if pd.notna(value)]
    if not finite_values:
        return default
    return float(reducer(finite_values))


def convergence_epoch_from_history(history, fraction=CONVERGENCE_FRACTION):
    epochs = history.get("epoch", [])
    losses = history.get("test_loss", [])
    pairs = [
        (int(epoch), float(loss))
        for epoch, loss in zip(epochs, losses)
        if pd.notna(loss) and np.isfinite(float(loss))
    ]
    if not pairs:
        return np.nan

    initial_loss = pairs[0][1]
    best_loss = min(loss for _, loss in pairs)
    improvement = initial_loss - best_loss
    if improvement <= 1e-12:
        return min(epoch for epoch, loss in pairs if loss == best_loss)

    threshold = initial_loss - fraction * improvement
    for epoch, loss in pairs:
        if loss <= threshold:
            return epoch
    return pairs[-1][0]


def make_summary_table(all_task_outputs):
    rows = []
    for task_key, outputs in all_task_outputs.items():
        task_config = TASK_CONFIGS[task_key]
        diagnostic = outputs["diagnostic_summary"].set_index("method")
        is_classification = task_config["task_type"] == "classification"
        score_metric = "Accuracy" if is_classification else "R2"
        train_score_key = "train_acc" if is_classification else "train_r2"
        test_score_key = "test_acc" if is_classification else "test_r2"

        for method in task_config["methods"]:
            result = outputs["results_by_method"].get(method)
            if result is None:
                continue

            history = result.get("history", {})
            if method == "bp":
                cosine_similarity = 1.0
                update_variance = 0.0
            elif method in diagnostic.index:
                cosine_similarity = float(diagnostic.loc[method, "cosine_similarity"])
                update_variance = float(diagnostic.loc[method, "update_variance"])
            else:
                cosine_similarity = np.nan
                update_variance = np.nan

            rows.append(
                {
                    "task": task_config["display_name"],
                    "method": METHOD_LABELS[method],
                    "best_train_loss": _best_history_value(history, "train_loss", min),
                    "best_test_loss": _best_history_value(history, "test_loss", min),
                    "best_train_score": _best_history_value(history, train_score_key, max),
                    "best_test_score": _best_history_value(history, test_score_key, max),
                    "score_metric": score_metric,
                    "convergence_epoch": convergence_epoch_from_history(history),
                    "cosine": cosine_similarity,
                    "variance": update_variance,
                }
            )
    return pd.DataFrame(rows)


NUMERIC_SUMMARY_COLUMNS = [
    "best_train_loss",
    "best_test_loss",
    "best_train_score",
    "best_test_score",
    "cosine",
    "variance",
]


def _format_scientific_sigfigs(value, significant_figures=SUMMARY_SIGNIFICANT_FIGURES):
    mantissa, exponent = f"{value:.{significant_figures - 1}e}".split("e")
    return f"{mantissa}e{int(exponent)}"


def format_summary_number(value):
    if pd.isna(value):
        return ""
    value = float(value)
    if value == 0.0:
        return "0.00"

    abs_value = abs(value)
    if abs_value < 1e-3 or abs_value >= 1e4:
        return _format_scientific_sigfigs(value)

    exponent = int(np.floor(np.log10(abs_value)))
    decimals = SUMMARY_SIGNIFICANT_FIGURES - exponent - 1
    rounded_value = round(value, decimals)
    rounded_abs = abs(rounded_value)

    if rounded_abs == 0.0:
        return "0.00"
    if rounded_abs < 1e-3 or rounded_abs >= 1e4:
        return _format_scientific_sigfigs(rounded_value)

    rounded_exponent = int(np.floor(np.log10(rounded_abs)))
    rounded_decimals = max(0, SUMMARY_SIGNIFICANT_FIGURES - rounded_exponent - 1)
    return f"{rounded_value:.{rounded_decimals}f}"


def format_summary_epoch(value):
    if pd.isna(value):
        return ""
    return str(int(round(float(value))))


def format_summary_table(summary_table):
    formatted = summary_table.copy()
    for column in NUMERIC_SUMMARY_COLUMNS:
        formatted[column] = formatted[column].map(format_summary_number)
    formatted["convergence_epoch"] = formatted["convergence_epoch"].map(format_summary_epoch)
    return formatted


summary_table = make_summary_table(all_task_outputs)
summary_table_formatted = format_summary_table(summary_table)
summary_raw_csv_path = TABLE_DIR / "results_summary_raw.csv"
summary_csv_path = TABLE_DIR / "results_summary.csv"
summary_tex_path = TABLE_DIR / "results_summary.tex"
summary_table.to_csv(summary_raw_csv_path, index=False)
summary_table_formatted.to_csv(summary_csv_path, index=False)
summary_table_formatted.to_latex(
    summary_tex_path,
    index=False,
    escape=True,
)

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)

print(f"Wrote raw numeric table: {summary_raw_csv_path}")
print(f"Wrote formatted table: {summary_csv_path}")
print(f"Wrote formatted LaTeX table: {summary_tex_path}")
print(f"Wrote downloadable archive: {zip_path}")
summary_table_formatted


## Optional Download Helper

In Colab, run the next cell if you want the output archive to download immediately instead of using the file browser.

In [ ]:

# Uncomment in Colab if you want an immediate browser download.
# from google.colab import files
# files.download(str(OUTPUT_DIR) + ".zip")
